# EduCare AI :  Final

**Project:** EduCare AI : RAG-based dual-mode medical assistant

**Module:** CMP2804M Team Software Engineering  Assessment 2

This is a  single end-to-end notebook covering the full system, evaluation harness,
and user-testing analysis.

## What this notebook does:

| Section | Purpose |
|---|---|
| 1. Setup | Install dependencies, mount Drive, define paths |
| 2. Corpus | Build the 30-document NHS + NICE corpus on disk |
| 3. Chunking | Boundary-aware splitter producing JSONL chunks |
| 4. Embeddings + FAISS | MiniLM embeddings, HNSW + exact dual indices |
| 5. Retrieval | Mode-filtered top-k with content-hash dedup |
| 6. Generation | flan-t5-large with mode-specific prompts |
| 7. Safety | Categorised regex classifier + patient-mode dose refusal |
| 8. Pipeline | End-to-end `educare_answer()` |
| 9. API | FastAPI with CORS, rate limiting, `/answer`, `/answer_eval`, `/feedback` |
| 10. Demo | Gradio UI + ngrok exposure |
| 11. Evaluation | 36-question gold bank, all proposal metrics |
| 12. User-testing analysis | Feedback ingestion + Report of Assessment |

## Notebook conventions:

- Each section is independent enough to re-run after a kernel restart,
  given the FAISS index and corpus persist to Drive.
- Comments explain *why* a choice was made, not *what* the code does.
- Section headers tell you what's coming out without reading the code.
- Heavy steps (corpus build, embedding, FAISS build, model load) cache
  to Drive so re-runs are seconds, not minutes.

## Model choice rationale :

| Sprint | Model | Reason |
|---|---|---|
| 2+3 | `flan-t5-base` (250M) | Baseline. Validated the RAG pipeline architecture and dual-mode separation. Accuracy and readability sat below proposal targets due to the small model's terseness.

---
## 1. Setup

Installs, Drive mount, project paths. Run this section once per Colab session.

In [ ]:
# Pinned to versions known to work together. slowapi for the rate-limit layer; pyngrok for the public-URL tunnel.
!pip install -q faiss-cpu sentence-transformers fastapi uvicorn pydantic tqdm transformers accelerate gradio nest-asyncio pyngrok requests slowapi textstat pandas matplotlib aiohttp

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os, json, hashlib, re, time, uuid
from pathlib import Path

# All persistent state lives under here. Re-running the notebook on a
# fresh kernel rebuilds from this directory in seconds.
BASE = "/content/drive/MyDrive/educare_ai"

PATHS = {
    "raw_nhs":   f"{BASE}/data/raw/nhs",
    "raw_nice":  f"{BASE}/data/raw/nice",
    "processed": f"{BASE}/data/processed",
    "index":     f"{BASE}/data/index",
    "feedback":  f"{BASE}/data/feedback",
    "eval":      f"{BASE}/data/evaluation",
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

CHUNK_FILE   = f"{PATHS['processed']}/chunks.jsonl"
HNSW_PATH    = f"{PATHS['index']}/faiss_hnsw.index"
EXACT_PATH   = f"{PATHS['index']}/faiss_exact.index"
META_PATH    = f"{PATHS['index']}/meta.jsonl"
FEEDBACK_LOG = f"{PATHS['feedback']}/feedback.jsonl"
SESSIONS_LOG = f"{PATHS['feedback']}/sessions.jsonl"

print(f"Project base: {BASE}")
print("Paths ready.")

Project base: /content/drive/MyDrive/educare_ai
Paths ready.



## 2. Corpus:

30 documents total: **15 NHS** patient-mode + **15 NICE** professional-mode.
Each carries a metadata header (`Title`, `Source`, `Mode`, `Topic`) so
mode-aware retrieval is enforced structurally.

The documents are structured summaries derived from publicly available
NHS and NICE pages. The references list cites the source URLs.

Patient-mode (NHS): Asthma, Diabetes, High Blood Pressure, Stroke, Heart Attack, High Cholesterol, Copd, Depression, Anxiety, Flu, Headache, Back Pain, Common Cold, Uti, Allergies.

Professional-mode (NICE): Hypertension, Stroke Tia, Asthma Chronic, Copd Management, Heart Failure, Depression Adults, Anxiety Gad, Type2 Diabetes, Type1 Diabetes, Atrial Fibrillation, Chronic Kidney Disease, Sepsis, Acute Kidney Injury, Pneumonia, Acs.

In [ ]:
# All 30 documents bundled into one cell so the corpus is a single,
# inspectable artefact rather than scattered across 30 cells. Each document is written to disk under data/raw/nhs or data/raw/nice.
CORPUS_PAYLOAD = [
  {
    "filename": "asthma.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Asthma\nSource: NHS\nMode: patient\nTopic: Asthma\n\nOverview:\nAsthma is a common condition that affects breathing. It cannot be cured, but with proper treatment most people can manage their symptoms effectively.\n\nSymptoms:\nCommon symptoms include wheezing, coughing, shortness of breath, and chest tightness. Symptoms may be mild or severe, can come and go, and are often worse at night or early in the morning.\n\nTriggers:\nAsthma symptoms can be triggered by exercise, air pollution, cold air, infections, smoke, or allergens such as pollen, dust, mould, or animals.\n\nWhen to seek medical advice:\nSee a GP if you or your child have asthma symptoms, if treatment is not helping, if inhaler use is becoming more frequent, or if symptoms affect daily activities or sleep.\n\nEmergency management:\nIf having an asthma attack, sit upright, stay calm, and use a reliever inhaler.\n- Blue inhaler: 1 puff every 30 to 60 seconds, up to 10 puffs\n- AIR or MART inhaler: 1 puff every 1 to 3 minutes, up to 6 puffs\nFollow a personal asthma action plan if it gives different advice.\n\nCall 999 if symptoms worsen, do not improve after the maximum inhaler dose, or if no inhaler is available. If symptoms are still no better after 10 minutes and help has not arrived, repeat inhaler use up to the same maximum dose.\n\nDiagnosis:\nDiagnosis may include clinical assessment, breathing tests, blood tests, and peak flow monitoring. Asthma may take time to diagnose because symptoms vary over time.\n\nTreatment:\nTreatment includes inhalers, symptom monitoring, avoiding triggers, and following an asthma action plan. Annual review is recommended.\n\nInhaler types:\nTreatment may include a reliever inhaler, a daily preventer inhaler, or a combination inhaler such as AIR or MART. Correct inhaler technique is important.\n\nLiving with asthma:\nMost people with asthma can lead normal active lives if their condition is well managed. Annual reviews and an up-to-date personal action plan support good control.\n"
  },
  {
    "filename": "diabetes.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Diabetes\nSource: NHS\nMode: patient\nTopic: Diabetes\n\nOverview:\nDiabetes is a condition that causes blood glucose (sugar) levels to become too high.\n\nTypes:\n- Type 1 diabetes: often diagnosed in childhood but can occur at any age and cannot be prevented\n- Type 2 diabetes: risk is higher with factors such as age, ethnicity, and overweight; it can sometimes be prevented or delayed\n- Gestational diabetes: occurs during pregnancy and usually goes away after birth\n\nSymptoms:\nCommon symptoms include feeling thirsty all the time, peeing more than usual, feeling very tired, and losing weight without trying.\n\nWhen to seek medical advice:\nSeek urgent medical advice or contact NHS 111 if you or your child has symptoms of diabetes. See a GP if you or your child are at higher risk of diabetes, even without symptoms.\n\nCauses:\nDiabetes happens when the body does not make enough insulin, makes no insulin, or cannot use insulin properly. Without effective insulin action, blood glucose levels rise.\n\nRisk factors:\nRisk factors for type 2 diabetes include weight, age, ethnicity, and family history.\n\nRisk reduction:\nRisk of type 2 and gestational diabetes may be reduced through a balanced diet, regular exercise, and maintaining a healthy weight.\n\nPre-diabetes:\nPre-diabetes means blood glucose levels are above normal but not high enough for diabetes diagnosis. It increases the risk of type 2 diabetes, but lifestyle changes can reduce this risk.\n\nTreatment:\nTreatment depends on the type of diabetes.\n- Type 1 diabetes: lifelong insulin therapy and regular blood glucose monitoring\n- Type 2 diabetes: lifestyle changes and sometimes medicines such as metformin or insulin\n- Gestational diabetes: lifestyle changes and sometimes medicines or insulin\n\nComplications:\nPoorly controlled diabetes can lead to heart attack, stroke, kidney problems, nerve damage, foot problems, and eye problems. Regular check-ups support early detection and prevention.\n\nLiving with diabetes:\nSelf-management includes monitoring blood glucose, eating a balanced diet, regular activity, and attending review appointments.\n"
  },
  {
    "filename": "high_blood_pressure.txt",
    "folder_key": "raw_nhs",
    "content": "Title: High Blood Pressure (Hypertension)\nSource: NHS\nMode: patient\nTopic: High blood pressure\n\nOverview:\nHigh blood pressure, also called hypertension, means your blood pressure is consistently higher than the recommended level. It often has no symptoms, but it raises the risk of heart attacks and strokes.\n\nSymptoms:\nHigh blood pressure usually has no symptoms. The only way to find out if you have it is to have your blood pressure checked.\n\nWhen to get checked:\nAll adults over 40 should have their blood pressure checked at least every 5 years. You can get a free check at your GP surgery, some pharmacies, or workplaces.\n\nCauses:\nThe exact cause of high blood pressure is often unknown, but the following can increase your risk:\n- being overweight\n- eating too much salt\n- not eating enough fruit and vegetables\n- not doing enough exercise\n- drinking too much alcohol or coffee\n- smoking\n- not getting enough sleep\n- being over 65\n- having a relative with high blood pressure\n- being of Black African or Black Caribbean descent\n\nLifestyle changes:\nYou can reduce high blood pressure by eating a healthy diet low in salt, exercising regularly, cutting down on alcohol, losing weight if overweight, and stopping smoking.\n\nTreatment:\nIf lifestyle changes are not enough, your GP may prescribe medicines to lower your blood pressure. You may need to take medicine for the rest of your life, but it can be reduced if your blood pressure stays low.\n\nRisks of untreated high blood pressure:\nUntreated high blood pressure can cause heart disease, heart attack, stroke, kidney disease, and vascular dementia.\n\nHome monitoring:\nYou can check your own blood pressure at home using a monitor. Your GP can advise on which monitor to use and how often to check.\n"
  },
  {
    "filename": "stroke.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Stroke\nSource: NHS\nMode: patient\nTopic: Stroke\n\nOverview:\nA stroke is a serious life-threatening medical condition that happens when the blood supply to part of the brain is cut off.\n\nRecognising a stroke (FAST):\nUse the FAST test to recognise the symptoms of a stroke:\n- Face: has the face fallen on one side? Can the person smile?\n- Arms: can the person raise both arms and keep them there?\n- Speech: is their speech slurred?\n- Time: it is time to call 999 immediately if you notice any of these signs.\n\nOther symptoms:\nOther symptoms include sudden weakness or numbness on one side of the body, sudden loss of vision, sudden confusion, dizziness, or a sudden severe headache.\n\nWhen to seek emergency help:\nCall 999 immediately if you or someone else has any symptoms of a stroke. Do not wait to see if symptoms improve.\n\nCauses:\nThe two main causes of stroke are ischaemic stroke (a blocked blood vessel) and haemorrhagic stroke (a burst blood vessel). A transient ischaemic attack (TIA), or \"mini-stroke\", happens when the blood supply to the brain is temporarily interrupted.\n\nRisk factors:\nRisk factors include high blood pressure, smoking, diabetes, high cholesterol, atrial fibrillation, being overweight, drinking too much alcohol, and a family history of stroke.\n\nTreatment:\nTreatment depends on the type of stroke. Ischaemic strokes may be treated with clot-busting medicines or procedures to remove the clot. Haemorrhagic strokes may need surgery.\n\nRecovery:\nRecovery from a stroke can take months or years. Rehabilitation may include physiotherapy, speech therapy, and occupational therapy. Some people make a full recovery; others have long-term effects.\n\nPreventing a stroke:\nReduce your risk by managing blood pressure, eating a healthy diet, exercising regularly, not smoking, drinking alcohol within recommended limits, and managing other conditions such as diabetes.\n"
  },
  {
    "filename": "heart_attack.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Heart Attack\nSource: NHS\nMode: patient\nTopic: Heart attack\n\nOverview:\nA heart attack is a serious medical emergency in which the supply of blood to the heart is suddenly blocked, usually by a blood clot.\n\nSymptoms:\nSymptoms of a heart attack can include:\n- chest pain that may feel like pressure, tightness, or squeezing\n- pain spreading to the arms, jaw, neck, back, or stomach\n- shortness of breath\n- feeling weak, light-headed, or both\n- feeling sick (nausea) or being sick\n- feeling sweaty\n- a feeling of overwhelming anxiety\n\nSymptoms in women:\nWomen are more likely than men to have less obvious symptoms such as nausea, fatigue, and back or jaw pain.\n\nWhen to call 999:\nCall 999 immediately if you or someone else may be having a heart attack. Every minute matters.\n\nWhile waiting for an ambulance:\n- sit down and rest\n- chew and swallow one adult aspirin (300 mg) if you are not allergic to aspirin\n- stay calm\n\nCauses:\nA heart attack is usually caused by coronary heart disease, in which the coronary arteries become narrowed by a build-up of fatty material called plaque. If a piece of plaque breaks off, a blood clot can form and block the artery.\n\nRecovery:\nMost people who survive a heart attack can return to a normal life with the right treatment and lifestyle changes. Cardiac rehabilitation programmes support recovery.\n\nPreventing a heart attack:\nReduce your risk by eating a healthy diet, exercising regularly, not smoking, keeping your weight in a healthy range, managing blood pressure and cholesterol, and limiting alcohol.\n"
  },
  {
    "filename": "high_cholesterol.txt",
    "folder_key": "raw_nhs",
    "content": "Title: High Cholesterol\nSource: NHS\nMode: patient\nTopic: High cholesterol\n\nOverview:\nCholesterol is a fatty substance found in the blood. Having high cholesterol does not usually cause any symptoms, but it raises the risk of heart problems and stroke.\n\nSymptoms:\nHigh cholesterol does not cause any symptoms. The only way to know if you have it is to get a blood test.\n\nCauses:\nHigh cholesterol can be caused by eating fatty food, not exercising enough, being overweight, smoking, and drinking alcohol. It can also run in families.\n\nTesting:\nYour GP may recommend a cholesterol blood test if you are over 40, are overweight, or have a family history of high cholesterol or heart problems. Some tests need fasting beforehand; your GP will advise.\n\nHealthy eating:\nTo lower cholesterol, eat less saturated fat (in fatty meats, butter, cheese, biscuits, cakes), eat more oily fish, nuts, seeds, fruit, and vegetables, and replace saturated fats with unsaturated fats such as olive oil.\n\nExercise:\nAim for at least 150 minutes of moderate-intensity activity each week, such as walking, swimming, or cycling.\n\nOther lifestyle steps:\nStop smoking, drink alcohol within recommended limits, and try to reach a healthy weight.\n\nTreatment:\nIf lifestyle changes are not enough, your GP may prescribe medicines such as statins. Statins are usually taken for life, and you should not stop them without advice.\n\nSide effects of statins:\nMost people taking statins do not have side effects. Some people experience headaches, muscle pain, or digestive issues. Speak to your GP if you have side effects.\n"
  },
  {
    "filename": "copd.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Chronic Obstructive Pulmonary Disease (COPD)\nSource: NHS\nMode: patient\nTopic: COPD\n\nOverview:\nChronic obstructive pulmonary disease (COPD) is a group of lung conditions that cause breathing difficulties. The main types are emphysema and chronic bronchitis.\n\nSymptoms:\nCommon symptoms of COPD include:\n- increasing breathlessness, especially when active\n- a persistent cough that brings up phlegm\n- frequent chest infections\n- wheezing\n\nWhen to see a GP:\nSee a GP if you have persistent symptoms, especially if you are over 35 and smoke or used to smoke.\n\nCauses:\nCOPD is usually caused by long-term exposure to substances that damage the lungs, most commonly cigarette smoke. Air pollution, dust, and chemicals at work can also cause COPD.\n\nDiagnosis:\nCOPD is usually diagnosed with a breathing test called spirometry, which measures how well your lungs work.\n\nTreatment:\nThere is no cure for COPD, but treatment can help slow its progression and control symptoms. Treatment includes:\n- stopping smoking\n- inhalers and medicines\n- pulmonary rehabilitation\n- surgery or a lung transplant in rare cases\n\nFlare-ups:\nA flare-up (or exacerbation) is when symptoms suddenly get worse. It may be caused by a chest infection. You may need extra inhalers, antibiotics, or steroids during a flare-up.\n\nLiving with COPD:\nMany people can manage their condition through stopping smoking, exercise, vaccination against flu and pneumonia, and a balanced diet.\n"
  },
  {
    "filename": "depression.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Depression in Adults\nSource: NHS\nMode: patient\nTopic: Depression\n\nOverview:\nDepression is more than simply feeling unhappy or fed up for a few days. Most people go through periods of feeling down, but when you are depressed you feel persistently sad for weeks or months, rather than just a few days.\n\nSymptoms:\nSymptoms can include:\n- continuous low mood or sadness\n- feeling hopeless and helpless\n- low self-esteem\n- feeling tearful\n- feeling guilt-ridden\n- feeling irritable and intolerant of others\n- having no motivation or interest in things\n- finding it difficult to make decisions\n- not getting any enjoyment out of life\n- feeling anxious or worried\n- having suicidal thoughts or thoughts of harming yourself\n\nPhysical symptoms:\n- moving or speaking more slowly than usual\n- changes in appetite or weight\n- constipation\n- unexplained aches and pains\n- lack of energy\n- low sex drive\n- changes to your menstrual cycle\n- disturbed sleep\n\nWhen to seek help:\nSeek help from a GP if you experience symptoms of depression for most of the day, every day, for more than 2 weeks. If you have thoughts of self-harm or suicide, contact a GP, NHS 111, or the Samaritans (116 123) immediately.\n\nCauses:\nDepression can have many different triggers. It is often caused by a combination of factors, including stressful life events, family history, personality, illness, alcohol and drugs, or childbirth.\n\nTreatment:\nTreatment for depression usually involves a combination of self-help, talking therapies, and medicines. Mild depression may improve with self-help. Moderate to severe depression usually requires talking therapy (such as cognitive behavioural therapy) and antidepressant medicines.\n\nSelf-help:\nStay active, talk to someone you trust, eat well, avoid alcohol, and try to keep a regular sleep pattern.\n"
  },
  {
    "filename": "anxiety.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Generalised Anxiety Disorder in Adults\nSource: NHS\nMode: patient\nTopic: Anxiety\n\nOverview:\nGeneralised anxiety disorder (GAD) is a long-term condition that causes you to feel anxious about a wide range of situations and issues, rather than one specific event.\n\nSymptoms:\nSymptoms can be psychological and physical. Psychological symptoms include:\n- restlessness\n- a sense of dread\n- feeling constantly \"on edge\"\n- difficulty concentrating\n- irritability\n\nPhysical symptoms include:\n- dizziness\n- tiredness\n- heart palpitations\n- muscle aches and tension\n- trembling or shaking\n- dry mouth\n- excessive sweating\n- shortness of breath\n- stomach ache\n- headache\n\nWhen to see a GP:\nSee a GP if anxiety is affecting your daily life or causing you distress.\n\nCauses:\nThe exact cause of GAD is not fully understood. It is likely a combination of factors, including overactivity in areas of the brain involved in emotions and behaviour, an imbalance of brain chemicals, genes inherited from your parents, having a history of stressful or traumatic experiences, having a painful long-term health condition, or having a history of drug or alcohol misuse.\n\nTreatment:\nTreatment options include:\n- talking therapies, such as cognitive behavioural therapy (CBT)\n- medicines, such as a type of antidepressant called SSRIs\n- self-help resources, such as workbooks or online courses\n\nSelf-help:\nThings you can try yourself include:\n- regular exercise\n- relaxation techniques such as breathing exercises\n- limiting caffeine and alcohol\n- talking to someone you trust\n- keeping a worry diary\n\nWhen to seek urgent help:\nContact NHS 111 or your GP urgently if anxiety is causing severe distress or you are having thoughts of self-harm.\n"
  },
  {
    "filename": "flu.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Flu (Influenza)\nSource: NHS\nMode: patient\nTopic: Flu\n\nOverview:\nFlu, or influenza, is a common infectious viral illness that spreads through coughs and sneezes. Most people recover within a week, but flu can sometimes cause serious complications.\n\nSymptoms:\nFlu symptoms come on very quickly and can include:\n- a sudden high temperature\n- an aching body\n- feeling tired or exhausted\n- a dry cough\n- a sore throat\n- a headache\n- difficulty sleeping\n- loss of appetite\n- diarrhoea or stomach pain\n- feeling sick and being sick\n\nDifference between flu and a cold:\nFlu symptoms come on quickly and tend to be more severe than a cold. Flu makes you feel exhausted; a cold usually does not.\n\nHow to treat flu yourself:\nMost people can treat flu at home by:\n- resting and sleeping\n- keeping warm\n- drinking plenty of water to avoid dehydration\n- taking paracetamol or ibuprofen to lower a high temperature and ease aches\n\nWhen to see a GP or call 111:\nContact NHS 111 if you:\n- are worried about your baby's or child's symptoms\n- are 65 or over\n- are pregnant\n- have a long-term medical condition such as diabetes or a heart, lung, or kidney condition\n- have a weakened immune system\n- have symptoms that do not improve after 7 days\n\nWhen to call 999:\nCall 999 or go to A&E if you or someone else has:\n- sudden chest pain\n- difficulty breathing\n- coughing up blood\n\nFlu vaccination:\nThe flu vaccine is offered free on the NHS to people at higher risk, including older adults, pregnant women, children, and people with long-term conditions. Vaccination reduces the risk of getting flu and of serious illness if you do.\n"
  },
  {
    "filename": "headache.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Headaches\nSource: NHS\nMode: patient\nTopic: Headache\n\nOverview:\nMost headaches are not serious and can be treated at home. Knowing the type of headache you have can help you get the right treatment.\n\nCommon types of headache:\n- Tension headache: the most common type, often described as a constant ache on both sides of the head, sometimes with tightness in the neck.\n- Migraine: a moderate or severe headache, usually on one side, that may come with feeling sick, being sick, and sensitivity to light or sound.\n- Cluster headache: severe pain on one side of the head, often around the eye, that comes in clusters over weeks or months.\n- Medication overuse headache: caused by taking painkillers too often.\n\nWhen to see a GP:\nSee a GP if your headache:\n- keeps coming back\n- is getting worse\n- is not relieved by painkillers\n- affects your daily life\n\nWhen to seek emergency help:\nCall 999 or go to A&E if you have a headache that:\n- comes on suddenly and is extremely severe (a \"thunderclap\" headache)\n- follows a head injury\n- comes with weakness, numbness, slurred speech, confusion, or loss of consciousness\n- comes with seizures\n- comes with a stiff neck, fever, rash, vomiting, or sensitivity to light\n\nSelf-help:\n- drink plenty of water\n- get plenty of rest if you also have a cold or the flu\n- try to relax and reduce stress\n- take paracetamol or ibuprofen\n- try to stay active\n\nWhen to avoid painkillers:\nAvoid taking painkillers more than 2 to 3 days a week, as this can cause medication overuse headaches.\n\nMigraine triggers:\nCommon triggers include stress, tiredness, certain foods or drinks, missed meals, hormonal changes, and bright lights.\n"
  },
  {
    "filename": "back_pain.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Back Pain\nSource: NHS\nMode: patient\nTopic: Back pain\n\nOverview:\nBack pain, particularly lower back pain, is very common. It usually improves within a few weeks or months. There are several things you can do to help ease the pain.\n\nCommon causes:\nMost back pain is not caused by anything serious and is often the result of:\n- a strain or sprain\n- poor posture\n- standing or bending for long periods\n- lifting or carrying something incorrectly\n\nThings you can do:\n- stay as active as possible and try to continue your daily activities\n- try exercises and stretches for back pain\n- take painkillers such as ibuprofen\n- use a hot or cold compress for short-term relief\n\nThings to avoid:\n- do not stay in bed for long periods\n- avoid heavy lifting until the pain eases\n\nWhen to see a GP:\nSee a GP if your back pain:\n- does not start to improve within a few weeks\n- stops you doing day-to-day activities\n- is severe or getting worse over time\n- is causing you to worry or you are struggling to cope\n\nWhen to seek emergency help:\nCall 999 or go to A&E if you have back pain along with:\n- numbness or tingling around your genitals or buttocks\n- difficulty peeing\n- loss of bladder or bowel control\n- chest pain\n- a high temperature\n- weight loss for no reason\n- swelling or a deformity in your back\n- pain that is worse at night\n- pain that started after a serious accident, such as a car crash\n\nPreventing back pain:\nYou can reduce your risk of back pain by exercising regularly, lifting correctly, maintaining good posture, and keeping a healthy weight.\n"
  },
  {
    "filename": "common_cold.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Common Cold\nSource: NHS\nMode: patient\nTopic: Common cold\n\nOverview:\nA cold is a mild viral infection of the nose, throat, sinuses, and upper airways. It is very common and usually clears up on its own within a week or two.\n\nSymptoms:\nSymptoms of a cold come on gradually and can include:\n- a blocked or runny nose\n- a sore throat\n- headaches\n- muscle aches\n- coughs\n- sneezing\n- a raised temperature\n- pressure in your ears and face\n- loss of taste and smell\n\nHow long it lasts:\nSymptoms are usually worst during the first 2 to 3 days before they gradually start to improve. In adults and older children, they usually last about 7 to 10 days.\n\nHow to treat a cold yourself:\n- rest, drink plenty of fluids, and eat healthily\n- gargle salt water to soothe a sore throat (not for children)\n- take paracetamol or ibuprofen to lower a high temperature and ease aches\n\nHow to avoid spreading a cold:\n- wash your hands often with warm water and soap\n- use tissues to trap germs when you cough or sneeze\n- bin used tissues as quickly as possible\n\nWhen to see a GP or call 111:\nContact NHS 111 if:\n- your symptoms do not improve after 3 weeks\n- your symptoms get suddenly worse\n- you have chest pain\n- you find it hard to breathe\n- you develop complications, such as chest pain or coughing up bloodstained mucus\n\nAntibiotics:\nAntibiotics are not effective against viruses and are not used to treat a cold. They will only be prescribed if a bacterial infection develops.\n"
  },
  {
    "filename": "uti.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Urinary Tract Infections (UTIs)\nSource: NHS\nMode: patient\nTopic: Urinary tract infection\n\nOverview:\nUrinary tract infections (UTIs) affect the urinary tract, including the bladder, urethra, or kidneys. They are common, especially in women, and can usually be treated with antibiotics.\n\nSymptoms:\nSymptoms can include:\n- pain or a burning sensation when peeing\n- needing to pee more often than usual\n- needing to pee suddenly or more urgently\n- pee that looks cloudy\n- blood in your pee\n- lower tummy pain or pain in your back, just under the ribs\n- a high temperature, or feeling hot and shivery\n- a very low temperature below 36C\n\nSymptoms in older people:\nIn older people, symptoms may include changes in behaviour, such as severe confusion or agitation.\n\nWhen to see a GP or call 111:\nContact NHS 111 or your GP if you:\n- have symptoms of a UTI for the first time\n- have blood in your pee\n- have a high temperature, or feel hot and shivery\n- have a very low temperature\n- feel confused, drowsy, or have difficulty speaking\n- have not peed all day\n- have severe lower tummy pain\n- are pregnant and have any symptoms\n- are a man with symptoms of a UTI\n\nTreatment:\nA GP may prescribe a short course of antibiotics. Take the full course even if you start to feel better. Drink plenty of water and take painkillers such as paracetamol if needed.\n\nThings to do yourself:\n- drink plenty of fluids\n- pee as soon as you need to, do not hold it in\n- wipe from front to back when you go to the toilet\n- pee as soon as possible after sex\n\nWhen to call 999:\nCall 999 if you or someone else has signs of sepsis, such as severe shivering, very fast heartbeat, very fast breathing, or extreme confusion.\n"
  },
  {
    "filename": "allergies.txt",
    "folder_key": "raw_nhs",
    "content": "Title: Allergies\nSource: NHS\nMode: patient\nTopic: Allergies\n\nOverview:\nAn allergy is a reaction the body has to a particular food or substance. Allergies are very common, affecting more than 1 in 4 people in the UK at some point in their lives.\n\nCommon allergies:\n- pollen (hay fever)\n- dust mites\n- animal dander, such as from cats and dogs\n- food, such as nuts, fruit, eggs, or milk\n- insect bites and stings\n- medicines, including some antibiotics\n- latex\n- mould\n\nSymptoms:\nSymptoms vary depending on what you are allergic to and how you came into contact with it. They can include:\n- sneezing\n- a runny or blocked nose\n- red, itchy, watery eyes\n- wheezing and coughing\n- a red, itchy rash\n- worsening of asthma or eczema symptoms\n\nSevere allergic reaction (anaphylaxis):\nA severe allergic reaction is called anaphylaxis. Symptoms include:\n- swelling of the throat and tongue\n- difficulty breathing or breathing very fast\n- difficulty swallowing, tightness in the throat, or a hoarse voice\n- wheezing, coughing, or noisy breathing\n- feeling tired or confused\n- feeling faint, dizzy, or fainting\n- skin that feels cold to the touch\n- blue, grey, or pale skin, lips, or tongue\n\nAnaphylaxis is a medical emergency. Call 999 immediately and use an adrenaline auto-injector (such as an EpiPen) if available.\n\nWhen to see a GP:\nSee a GP if you think you or your child may be having an allergic reaction. They can refer you to an allergy specialist if needed.\n\nTreatment:\nTreatment for allergies depends on what you are allergic to. The most effective way to manage an allergy is to avoid the allergen. Medicines such as antihistamines, decongestants, and steroid creams or sprays can help with symptoms.\n\nLiving with allergies:\nCarrying an adrenaline auto-injector is essential if you have been prescribed one. Wearing a medical alert bracelet can also help in an emergency.\n"
  },
  {
    "filename": "hypertension.txt",
    "folder_key": "raw_nice",
    "content": "Title: Hypertension\nSource: NICE\nMode: professional\nTopic: Hypertension\n\nOverview:\nHypertension is persistently raised arterial blood pressure. It increases the risk of heart failure, coronary artery disease, stroke, chronic kidney disease, peripheral arterial disease, and vascular dementia.\n\nTypes:\n- Primary hypertension: around 90% of cases, with no identifiable cause\n- Secondary hypertension: around 10% of cases, caused by renal, endocrine, vascular, or drug-related factors\n\nDiagnosis:\nSuspect hypertension if clinic systolic blood pressure is 140 mmHg or above and/or clinic diastolic blood pressure is 90 mmHg or above. Diagnosis should be confirmed using ambulatory blood pressure monitoring (ABPM) or home blood pressure monitoring (HBPM).\n\nInitial assessment:\nWhile awaiting confirmation:\n- investigate for target organ damage\n- assess for secondary causes\n- evaluate cardiovascular risk\n\nClassification:\n- Stage 1 hypertension:\n  - clinic BP 140/90 to 159/99 mmHg\n  - ABPM/HBPM 135/85 to 149/94 mmHg\n- Stage 2 hypertension:\n  - clinic BP 160/100 to 179/119 mmHg\n  - ABPM/HBPM 150/95 mmHg or above\n- Stage 3 or severe hypertension:\n  - clinic systolic 180 mmHg or above or diastolic 120 mmHg or above\n\nSpecial types:\n- Accelerated hypertension: BP 180/120 mmHg or above with retinal haemorrhage or papilloedema\n- White-coat hypertension: raised BP in clinic but normal outside clinic\n\nUrgent referral:\nArrange same-day specialist assessment if BP is 180/120 mmHg or above with retinal haemorrhage, papilloedema, confusion, chest pain, signs of heart failure, or acute kidney injury, or if phaeochromocytoma is suspected.\n\nSpecialist consideration:\nIn adults under 40 years with hypertension, consider specialist evaluation for secondary causes and long-term treatment balance.\n\nManagement:\nManagement includes lifestyle advice, stepwise antihypertensive treatment, cardiovascular risk reduction, and monitoring.\n\nTreatment targets:\n- Adults under 80: clinic BP target below 140/90 mmHg, ABPM/HBPM below 135/85 mmHg\n- Adults 80 and over: clinic BP target below 150/90 mmHg, ABPM/HBPM below 145/85 mmHg\n\nStepwise treatment:\n- Step 1: ACE inhibitor or ARB (under 55 and not of African or African-Caribbean family origin) or calcium channel blocker (over 55 or of African or African-Caribbean family origin)\n- Step 2: combine an ACE inhibitor or ARB with a calcium channel blocker or thiazide-like diuretic\n- Step 3: add the third agent to achieve A+C+D\n- Step 4 (resistant hypertension): consider further diuretic, alpha-blocker, or beta-blocker; seek specialist advice\n\nLifestyle advice:\nReduce dietary salt, moderate alcohol intake, encourage regular exercise, weight management, and smoking cessation.\n"
  },
  {
    "filename": "stroke_tia.txt",
    "folder_key": "raw_nice",
    "content": "Title: Stroke and Transient Ischaemic Attack (TIA)\nSource: NICE\nMode: professional\nTopic: Stroke and TIA\n\nOverview:\nStroke is a vascular clinical syndrome causing rapidly developing focal or global neurological dysfunction lasting more than 24 hours or leading to death. Transient ischaemic attack (TIA) is a transient focal neurological dysfunction lasting less than 24 hours without evidence of acute infarction.\n\nEpidemiology:\nAround 85% of strokes are ischaemic and 15% are haemorrhagic. Stroke is a major cause of death and disability in the UK.\n\nClinical presentation:\nStroke and TIA usually present with sudden onset focal neurological symptoms such as weakness, numbness, slurred speech, or visual disturbance. Symptoms should not be better explained by another condition such as hypoglycaemia.\n\nDifferential diagnosis:\nConsider hypoglycaemia, seizure, migraine with aura, syncope, peripheral vestibular disorders, and functional neurological disorder.\n\nAcute stroke management:\n- arrange immediate emergency admission to a stroke unit\n- provide advance information to ambulance control and the receiving hospital\n- avoid antiplatelet treatment until haemorrhagic stroke has been excluded\n- thrombolysis with alteplase may be offered within 4.5 hours of symptom onset for ischaemic stroke\n- thrombectomy may be considered for selected patients with large vessel occlusion\n\nSuspected TIA management:\n- give aspirin 300 mg immediately unless contraindicated or the person is already taking aspirin regularly\n- arrange specialist assessment within 24 hours if the suspected TIA occurred within the last week\n- arrange specialist assessment within 7 days if the suspected TIA occurred more than 1 week ago\n- advise the person not to drive until specialist guidance is given\n\nInvestigations:\n- urgent brain imaging (CT or MRI) to differentiate ischaemic from haemorrhagic stroke\n- carotid imaging in suspected anterior circulation events\n- ECG and 24-hour cardiac monitoring to detect atrial fibrillation\n- bloods including FBC, U&E, glucose, lipids, clotting\n\nSecondary prevention:\n- antiplatelet therapy for ischaemic stroke or TIA (clopidogrel preferred long-term)\n- statin therapy\n- blood pressure management\n- anticoagulation if atrial fibrillation is identified\n- lifestyle advice including smoking cessation, diet, and exercise\n\nRehabilitation:\nMultidisciplinary rehabilitation should address physical, communication, cognitive, and psychological needs. Early supported discharge may be appropriate for selected patients.\n\nFollow-up:\nArrange follow-up on discharge, at 6 months, and then annually. Address secondary prevention, rehabilitation needs, mood, and cognition.\n"
  },
  {
    "filename": "asthma_chronic.txt",
    "folder_key": "raw_nice",
    "content": "Title: Asthma — Diagnosis, Monitoring and Chronic Asthma Management\nSource: NICE\nMode: professional\nTopic: Asthma chronic management\n\nOverview:\nAsthma is a chronic inflammatory disorder of the airways characterised by variable, reversible airflow obstruction and airway hyperresponsiveness.\n\nDiagnostic criteria:\nDiagnose asthma in adults if there is a history of episodic symptoms (wheeze, breathlessness, cough, chest tightness) and objective evidence from at least one of:\n- FeNO of 50 ppb or more (40 ppb or more in children)\n- bronchodilator reversibility with FEV1 increase of 12% or more\n- peak expiratory flow variability over 20% across 2 weeks\n- positive bronchial challenge test (where available)\n\nPharmacological management (adults and children 12 and over):\n- Step 1: low-dose ICS-formoterol as needed (AIR therapy)\n- Step 2: low-dose maintenance plus reliever therapy (MART)\n- Step 3: moderate-dose MART\n- Step 4: refer to specialist; consider FeNO, eosinophils, and add-on therapy\n- Step 5: specialist-led biological therapies\n\nInhaler choice:\nChoose inhalers based on patient ability, preference, and environmental impact. Provide training in inhaler technique at every consultation.\n\nAdherence and review:\nAnnual review is recommended. Assess symptoms, exacerbation history, inhaler technique, adherence, and consider step-down if well controlled for 3 or more months.\n\nPersonalised asthma action plan:\nProvide all patients with a written personalised asthma action plan covering routine treatment, recognising deterioration, and emergency action.\n\nAcute asthma in adults:\nSeverity assessment:\n- Moderate: PEF 50-75% best/predicted, no features of severe asthma\n- Severe: PEF 33-50%, RR 25 or more, HR 110 or more, inability to complete sentences\n- Life-threatening: PEF below 33%, SpO2 below 92%, silent chest, cyanosis, exhaustion, hypotension, arrhythmia, altered consciousness\n\nTreatment of acute severe asthma:\n- oxygen to maintain SpO2 94-98%\n- nebulised salbutamol 5 mg\n- prednisolone 40-50 mg orally (or IV hydrocortisone if not tolerated)\n- nebulised ipratropium for severe or life-threatening attacks\n- escalate to senior support and consider IV magnesium sulphate\n\nDischarge criteria:\nStable on discharge medication, PEF over 75% best/predicted, inhaler technique checked, oral steroids prescribed, follow-up arranged within 2 working days.\n"
  },
  {
    "filename": "copd_management.txt",
    "folder_key": "raw_nice",
    "content": "Title: Chronic Obstructive Pulmonary Disease in Over 16s\nSource: NICE\nMode: professional\nTopic: COPD\n\nOverview:\nCOPD is a heterogeneous condition characterised by persistent respiratory symptoms and airflow obstruction due to abnormalities of the airways and alveoli.\n\nDiagnostic criteria:\nSuspect COPD in people over 35 who have a risk factor (usually smoking) and present with one or more of: exertional breathlessness, chronic cough, regular sputum production, frequent winter bronchitis, or wheeze. Confirm with post-bronchodilator spirometry showing FEV1/FVC ratio below 0.7.\n\nSeverity assessment:\nSeverity of airflow obstruction (post-bronchodilator FEV1 percent predicted):\n- Stage 1 mild: FEV1 80% or above\n- Stage 2 moderate: FEV1 50-79%\n- Stage 3 severe: FEV1 30-49%\n- Stage 4 very severe: FEV1 below 30%\n\nInitial management:\n- offer smoking cessation support at every contact\n- offer annual influenza and pneumococcal vaccination\n- offer pulmonary rehabilitation to all functionally limited patients (MRC dyspnoea grade 3 or above)\n\nInhaled therapy:\n- Step 1: short-acting bronchodilator (SABA or SAMA) as needed\n- Step 2: if asthmatic features or features suggesting steroid responsiveness, offer LABA + ICS; otherwise LABA + LAMA\n- Step 3: triple therapy LABA + LAMA + ICS\n\nExacerbations:\nTreat with short-acting bronchodilators, oral corticosteroids (prednisolone 30 mg for 5 days), and oral antibiotics if sputum is purulent or there are clinical signs of pneumonia.\n\nLong-term oxygen therapy:\nOffer LTOT to people with PaO2 below 7.3 kPa when stable, or below 8 kPa with secondary polycythaemia, peripheral oedema, or pulmonary hypertension. Counsel about the dangers of smoking with oxygen.\n\nSelf-management:\nProvide a personalised self-management plan, including a rescue pack of antibiotics and corticosteroids for selected patients.\n"
  },
  {
    "filename": "heart_failure.txt",
    "folder_key": "raw_nice",
    "content": "Title: Chronic Heart Failure in Adults\nSource: NICE\nMode: professional\nTopic: Heart failure\n\nOverview:\nHeart failure is a clinical syndrome characterised by typical symptoms and signs caused by structural or functional cardiac abnormalities, resulting in reduced cardiac output or elevated intracardiac pressures.\n\nClassification:\n- Heart failure with reduced ejection fraction (HFrEF): LVEF below 40%\n- Heart failure with mildly reduced ejection fraction (HFmrEF): LVEF 41-49%\n- Heart failure with preserved ejection fraction (HFpEF): LVEF 50% or above\n\nDiagnosis:\nMeasure NT-proBNP in people with suspected heart failure:\n- NT-proBNP above 2000 ng/L: refer urgently for specialist assessment and echocardiography within 2 weeks\n- NT-proBNP 400-2000 ng/L: refer for assessment and echocardiography within 6 weeks\n- NT-proBNP below 400 ng/L: heart failure unlikely\n\nInitial pharmacological management of HFrEF:\nFirst-line: ACE inhibitor (or ARB if not tolerated) and a beta-blocker licensed for heart failure (bisoprolol, carvedilol, or nebivolol).\n\nSecond-line:\nAdd a mineralocorticoid receptor antagonist (MRA) such as spironolactone or eplerenone if symptoms persist.\n\nFurther options:\n- sacubitril/valsartan (in place of ACEi/ARB) under specialist advice\n- SGLT2 inhibitors (dapagliflozin or empagliflozin)\n- ivabradine in selected sinus rhythm patients with HR 75 or above\n- hydralazine plus nitrate, particularly in people of African or Caribbean family origin\n\nDevice therapy:\nConsider cardiac resynchronisation therapy (CRT) and implantable cardioverter defibrillators (ICDs) according to LVEF, QRS duration, and symptoms.\n\nMonitoring:\nReview at least every 6 months in stable patients. Monitor symptoms, weight, fluid status, renal function, electrolytes, and medication tolerance.\n\nEnd-of-life care:\nConsider palliative care for people with advanced heart failure and refractory symptoms.\n"
  },
  {
    "filename": "depression_adults.txt",
    "folder_key": "raw_nice",
    "content": "Title: Depression in Adults — Treatment and Management\nSource: NICE\nMode: professional\nTopic: Depression\n\nOverview:\nDepression is a common mental disorder characterised by persistent sadness and a loss of interest in activities, with associated emotional, cognitive, physical, and behavioural symptoms.\n\nDiagnostic criteria:\nUse ICD-10 or DSM-5 criteria. Diagnose depression based on a comprehensive assessment that includes severity, duration, functional impairment, and the presence of suicidal ideation. Use validated tools such as PHQ-9 to assess severity.\n\nSeverity classification:\n- Less severe depression: PHQ-9 score below 16\n- More severe depression: PHQ-9 score 16 or above\n\nStepped care approach:\nChoose treatment in shared decision with the patient. Offer the least intrusive, most effective intervention first.\n\nLess severe depression — first-line options:\n- guided self-help\n- group cognitive behavioural therapy (CBT)\n- group behavioural activation\n- individual CBT\n- individual behavioural activation\n- group exercise\n- group mindfulness and meditation\n- interpersonal psychotherapy\n- SSRI antidepressants\n- counselling\n- short-term psychodynamic psychotherapy\n\nMore severe depression — first-line options:\n- a combination of individual CBT and antidepressant\n- individual CBT\n- individual behavioural activation\n- antidepressant medication (SSRI as first-line)\n- individual problem-solving\n- counselling\n- short-term psychodynamic psychotherapy\n- interpersonal psychotherapy\n- guided self-help\n\nAntidepressant choice:\nSSRIs are usually first-line. Consider patient preference, prior response, side-effect profile, drug interactions, comorbidities, and risk of overdose. Avoid combinations of antidepressants without specialist advice.\n\nReview:\nReview within 1 week of starting an antidepressant in people aged 18-25 or those at increased risk of suicide; otherwise within 2-4 weeks. Continue antidepressants for at least 6 months after remission.\n\nSuicide risk assessment:\nAt every contact, assess for suicidal ideation, intent, and plans. Provide a safety plan and arrange urgent assessment if there is significant risk.\n\nTreatment-resistant depression:\nConsider switching antidepressant, augmentation with lithium or a second-generation antipsychotic, or combination psychological and pharmacological treatment. Specialist referral may be required.\n"
  },
  {
    "filename": "anxiety_gad.txt",
    "folder_key": "raw_nice",
    "content": "Title: Generalised Anxiety Disorder and Panic Disorder in Adults\nSource: NICE\nMode: professional\nTopic: Generalised anxiety disorder\n\nOverview:\nGeneralised anxiety disorder (GAD) is characterised by excessive, persistent worry that is difficult to control and causes significant distress or impairment.\n\nDiagnostic criteria:\nUse ICD-10 or DSM-5 criteria. Symptoms must be present on most days for at least 6 months. Use validated tools such as the GAD-7 to support diagnosis and severity assessment.\n\nStepped care for GAD:\n- Step 1: identification, assessment, education, and active monitoring\n- Step 2: low-intensity psychological interventions (individual non-facilitated self-help, individual guided self-help, psychoeducational groups)\n- Step 3: high-intensity psychological intervention (CBT or applied relaxation) or drug treatment\n- Step 4: complex, treatment-refractory GAD — specialist mental health services, combination treatments, residential care\n\nDrug treatment:\nFirst-line: SSRI (sertraline is preferred where available based on cost-effectiveness). If sertraline is ineffective or not tolerated, consider another SSRI or an SNRI.\n\nPregabalin:\nConsider pregabalin if SSRIs and SNRIs are ineffective or not tolerated. Be aware of the risk of misuse and dependence.\n\nBenzodiazepines:\nDo not offer benzodiazepines for GAD except as a short-term measure during crises (no longer than 2-4 weeks).\n\nReview:\nReview within 2 weeks of starting drug treatment, then every 2-4 weeks for the first 3 months. Continue effective treatment for at least 12 months.\n\nPanic disorder:\nFor panic disorder, offer either CBT or an SSRI. If neither is acceptable, offer self-help based on CBT principles. Do not offer benzodiazepines for panic disorder.\n\nWhen to refer:\nRefer to specialist mental health services if there is significant comorbidity, risk to self, or treatment resistance after 2 step-3 interventions.\n"
  },
  {
    "filename": "type2_diabetes.txt",
    "folder_key": "raw_nice",
    "content": "Title: Type 2 Diabetes in Adults — Management\nSource: NICE\nMode: professional\nTopic: Type 2 diabetes\n\nOverview:\nType 2 diabetes is a chronic metabolic disorder characterised by insulin resistance and progressive beta-cell dysfunction, leading to hyperglycaemia.\n\nDiagnosis:\nDiagnose type 2 diabetes if:\n- HbA1c is 48 mmol/mol (6.5%) or above\n- fasting plasma glucose is 7.0 mmol/L or above\n- random plasma glucose is 11.1 mmol/L or above with symptoms\n\nConfirm with a repeat test in asymptomatic individuals.\n\nHbA1c targets:\n- Lifestyle alone or with metformin: aim for 48 mmol/mol (6.5%)\n- On a drug associated with hypoglycaemia: aim for 53 mmol/mol (7.0%)\n- Relax targets in older adults or those with significant comorbidities\n\nFirst-line drug therapy:\nOffer metformin (standard release initially, then consider modified release if GI side effects). Confirm cardiovascular risk; offer an SGLT2 inhibitor with metformin if there is established cardiovascular disease, heart failure, or chronic kidney disease.\n\nSecond-line therapy:\nIf HbA1c remains above the individual target after metformin, intensify with one of:\n- DPP-4 inhibitor\n- pioglitazone\n- sulphonylurea\n- SGLT2 inhibitor (preferred if CV/renal/HF risk)\n- GLP-1 receptor agonist (consider for patients with obesity)\n\nTriple therapy:\nCombine metformin with two of the above agents. Consider insulin if combination oral therapy is ineffective.\n\nInsulin therapy:\nInitiate basal insulin (NPH or analogue) when oral combinations fail to achieve target. Educate on self-monitoring, hypoglycaemia recognition, and dose titration.\n\nAnnual review:\nAnnual review should include HbA1c, blood pressure, lipids, weight, foot examination, retinal screening, urinary albumin-creatinine ratio, eGFR, and depression screening.\n\nCardiovascular risk:\nOffer atorvastatin 20 mg for primary prevention if QRISK is 10% or above. Treat hypertension to below 140/90 mmHg (below 130/80 mmHg if albuminuria or organ damage).\n"
  },
  {
    "filename": "type1_diabetes.txt",
    "folder_key": "raw_nice",
    "content": "Title: Type 1 Diabetes in Adults — Diagnosis and Management\nSource: NICE\nMode: professional\nTopic: Type 1 diabetes\n\nOverview:\nType 1 diabetes is an autoimmune disorder resulting in absolute insulin deficiency, requiring lifelong insulin replacement therapy.\n\nDiagnosis:\nSuspect type 1 diabetes in adults presenting with hyperglycaemia accompanied by:\n- ketosis\n- rapid weight loss\n- age below 50 at onset\n- BMI below 25\n- personal or family history of autoimmune disease\n\nInvestigations:\n- C-peptide measurement\n- diabetes-specific autoantibodies (GAD, IA-2, ZnT8)\n- ketones (blood or urine)\n\nInsulin regimens:\nOffer multiple daily injection (MDI) basal-bolus insulin as the regimen of choice. The basal insulin should be twice-daily insulin detemir; alternatives include once-daily glargine or degludec.\n\nContinuous subcutaneous insulin infusion (CSII):\nConsider CSII (insulin pump therapy) for adults with disabling hypoglycaemia or HbA1c above 69 mmol/mol despite optimised MDI.\n\nContinuous glucose monitoring:\nOffer real-time continuous glucose monitoring (rtCGM) or intermittently scanned CGM (isCGM) to all adults with type 1 diabetes.\n\nHbA1c target:\nAim for HbA1c of 48 mmol/mol (6.5%) or below in most adults. Avoid this target if it would cause significant hypoglycaemia.\n\nSelf-management education:\nOffer structured education programmes such as DAFNE (Dose Adjustment for Normal Eating) within 6-12 months of diagnosis.\n\nHypoglycaemia management:\nEducate patients on recognition and treatment of hypoglycaemia. Provide glucagon and educate family/carers on its administration. Refer to specialist services for problematic hypoglycaemia.\n\nDiabetic ketoacidosis (DKA):\nSuspect DKA in any unwell person with type 1 diabetes. Diagnostic criteria: blood glucose above 11 mmol/L or known diabetes, blood ketones 3 mmol/L or above (or urine ketones 2+ or above), bicarbonate below 15 mmol/L or pH below 7.3. Manage with fluid replacement, fixed-rate IV insulin infusion, and electrolyte correction.\n\nAnnual review:\nInclude HbA1c, blood pressure, lipids, retinal screening, foot examination, renal function, mental wellbeing, and screening for autoimmune comorbidities (thyroid, coeliac).\n"
  },
  {
    "filename": "atrial_fibrillation.txt",
    "folder_key": "raw_nice",
    "content": "Title: Atrial Fibrillation\nSource: NICE\nMode: professional\nTopic: Atrial fibrillation\n\nOverview:\nAtrial fibrillation (AF) is a supraventricular tachyarrhythmia characterised by uncoordinated atrial activation and ineffective atrial contraction. It significantly increases stroke risk.\n\nClassification:\n- Paroxysmal: episodes terminate spontaneously within 7 days\n- Persistent: episodes last longer than 7 days or require cardioversion\n- Long-standing persistent: continuous AF for more than 12 months\n- Permanent: AF is accepted; no further attempts at rhythm control\n\nDiagnosis:\nConfirm AF with a 12-lead ECG. In suspected paroxysmal AF, use ambulatory ECG monitoring (24-hour Holter, 7-day monitor, or implantable loop recorder).\n\nStroke risk assessment:\nAssess stroke risk using CHA2DS2-VASc score. Offer anticoagulation if score is 2 or above (or 1 or above in men); consider in men with score 1.\n\nBleeding risk assessment:\nUse ORBIT score. Modifiable bleeding risk factors should be addressed but should not preclude anticoagulation in eligible patients.\n\nAnticoagulation:\nFirst-line: a direct oral anticoagulant (DOAC) — apixaban, dabigatran, edoxaban, or rivaroxaban. Use a vitamin K antagonist (warfarin) if DOACs are unsuitable.\n\nRate control:\nFirst-line for most: beta-blocker (not sotalol) or rate-limiting calcium channel blocker (diltiazem or verapamil). Digoxin monotherapy is appropriate only for non-paroxysmal AF in sedentary patients.\n\nRhythm control:\nConsider rhythm control if symptoms persist despite rate control, or if AF is new-onset, has a reversible cause, is causing heart failure, or there is patient preference.\n\nCardioversion:\nOffer pharmacological or electrical cardioversion for AF lasting longer than 48 hours after at least 3 weeks of therapeutic anticoagulation, or after exclusion of left atrial thrombus by transoesophageal echocardiography.\n\nCatheter ablation:\nConsider catheter ablation for paroxysmal or persistent symptomatic AF when drug treatment is unsuccessful, contraindicated, or not tolerated.\n"
  },
  {
    "filename": "chronic_kidney_disease.txt",
    "folder_key": "raw_nice",
    "content": "Title: Chronic Kidney Disease — Assessment and Management\nSource: NICE\nMode: professional\nTopic: Chronic kidney disease\n\nOverview:\nChronic kidney disease (CKD) is defined by abnormalities of kidney structure or function present for more than 3 months, with implications for health.\n\nClassification:\nStage based on eGFR (mL/min/1.73 m²):\n- G1: 90 or above with markers of kidney damage\n- G2: 60-89 with markers of kidney damage\n- G3a: 45-59\n- G3b: 30-44\n- G4: 15-29\n- G5: below 15 (kidney failure)\n\nACR categories:\n- A1: ACR below 3 mg/mmol\n- A2: ACR 3-30 mg/mmol\n- A3: ACR above 30 mg/mmol\n\nTesting:\nUse eGFR calculated with the CKD-EPI creatinine equation. Confirm reduced eGFR with a repeat sample after at least 7 days. Measure ACR on an early morning urine sample.\n\nRisk stratification:\nUse eGFR and ACR together to estimate progression risk. Higher ACR confers higher risk at any given eGFR.\n\nManagement of CKD:\n- offer atorvastatin 20 mg for primary or secondary prevention of CV disease\n- treat hypertension to below 140/90 mmHg (below 130/80 mmHg if ACR 70 mg/mmol or above)\n- offer an ACE inhibitor or ARB to people with diabetes and ACR 3 mg/mmol or above, or hypertension and ACR 30 mg/mmol or above, or any CKD with ACR 70 mg/mmol or above\n- offer an SGLT2 inhibitor to people with CKD plus type 2 diabetes (and increasingly without diabetes for selected categories)\n\nReferral to nephrology:\nRefer when:\n- eGFR below 30 (or sustained decrease of 25% with category change)\n- ACR 70 mg/mmol or above\n- ACR 30 mg/mmol or above with haematuria\n- sustained decrease in eGFR of 15 mL/min/1.73 m² or more within 12 months\n- hypertension uncontrolled despite four agents\n- known or suspected rare or genetic causes\n- suspected renal artery stenosis\n\nLifestyle:\nSmoking cessation, weight management, regular exercise, dietary salt reduction, and avoidance of nephrotoxic drugs (especially NSAIDs).\n"
  },
  {
    "filename": "sepsis.txt",
    "folder_key": "raw_nice",
    "content": "Title: Sepsis — Recognition, Diagnosis and Early Management\nSource: NICE\nMode: professional\nTopic: Sepsis\n\nOverview:\nSepsis is a life-threatening organ dysfunction caused by a dysregulated host response to infection. Septic shock is sepsis with circulatory and cellular metabolic abnormalities.\n\nRecognition:\nSuspect sepsis in any person presenting with signs or symptoms suggesting possible infection. Risk-stratify using NICE risk-stratification tools, considering altered mental state, respiratory rate, oxygen saturations, blood pressure, heart rate, temperature, urine output, and skin signs.\n\nHigh-risk criteria (any one):\n- altered mental state (objective change from baseline)\n- RR 25 breaths/min or above\n- new oxygen requirement to maintain SpO2 92% or above\n- HR 130 bpm or above\n- systolic BP 90 mmHg or below, or more than 40 mmHg below normal\n- not passed urine in previous 18 hours (in adults)\n- mottled or ashen appearance\n- cyanosis\n- non-blanching petechial or purpuric rash\n\nInitial management of suspected sepsis (Sepsis Six within 1 hour):\n- give oxygen to maintain SpO2 94-98% (88-92% in COPD)\n- take blood cultures and consider source control\n- give broad-spectrum IV antibiotics\n- give IV fluids (crystalloid 30 mL/kg in suspected septic shock)\n- measure serial lactate\n- monitor urine output\n\nInvestigations:\nFBC, U&E, CRP, coagulation, LFTs, blood gas with lactate, blood cultures, urine culture, chest X-ray. Consider further imaging based on suspected source.\n\nAntibiotics:\nChoose antibiotics according to local protocols and likely source. Review within 24-48 hours and de-escalate based on culture results and clinical response.\n\nLactate:\nA lactate above 2 mmol/L indicates significant illness; above 4 mmol/L is associated with high mortality and warrants urgent senior review and ICU consideration.\n\nReassessment:\nReassess at 1 hour and 3 hours. Escalate to critical care if not improving despite initial resuscitation.\n\nSource control:\nIdentify and address the source of infection (drainage of abscesses, removal of infected catheters, surgery for surgical source) as soon as possible.\n"
  },
  {
    "filename": "acute_kidney_injury.txt",
    "folder_key": "raw_nice",
    "content": "Title: Acute Kidney Injury — Prevention, Detection and Management\nSource: NICE\nMode: professional\nTopic: Acute kidney injury\n\nOverview:\nAcute kidney injury (AKI) is a sudden reduction in kidney function characterised by a rise in serum creatinine, a fall in urine output, or both.\n\nDiagnostic criteria (KDIGO):\nDiagnose AKI if any of:\n- rise in serum creatinine of 26 micromol/L or more within 48 hours\n- 50% or more rise in creatinine known or presumed to have occurred within 7 days\n- fall in urine output to less than 0.5 mL/kg/hour for more than 6 hours\n\nStaging:\n- Stage 1: creatinine 1.5-1.9x baseline or 26 micromol/L or more rise; urine output below 0.5 mL/kg/h for 6-12 h\n- Stage 2: creatinine 2.0-2.9x baseline; urine output below 0.5 mL/kg/h for 12 h or more\n- Stage 3: creatinine 3.0x baseline or above 354 micromol/L; urine output below 0.3 mL/kg/h for 24 h or more, or anuria for 12 h\n\nRisk factors:\n- age 65 or above\n- pre-existing CKD\n- heart failure, liver disease, diabetes\n- nephrotoxic medications (NSAIDs, ACEi/ARB, diuretics, aminoglycosides)\n- recent contrast exposure\n- sepsis or hypovolaemia\n\nIdentification of cause:\nDistinguish pre-renal (hypovolaemia, hypotension, sepsis, heart failure), intrinsic (acute tubular necrosis, glomerulonephritis), and post-renal (obstruction).\n\nInitial management:\n- correct hypovolaemia with appropriate IV fluids\n- review and stop nephrotoxic medications\n- treat underlying cause (sepsis, obstruction)\n- monitor fluid balance, urine output, daily weights, U&Es\n\nIndications for renal replacement therapy:\n- refractory hyperkalaemia\n- severe metabolic acidosis\n- fluid overload unresponsive to diuretics\n- uraemic complications (encephalopathy, pericarditis)\n- specific intoxications\n\nReferral to nephrology:\nRefer if there is uncertain cause, evidence of intrinsic renal disease, complications, or stage 3 AKI not improving with initial management.\n\nFollow-up:\nPatients who have had AKI have an increased long-term risk of CKD; arrange follow-up with renal function checks 2-3 months post-event.\n"
  },
  {
    "filename": "pneumonia.txt",
    "folder_key": "raw_nice",
    "content": "Title: Pneumonia in Adults — Diagnosis and Management\nSource: NICE\nMode: professional\nTopic: Pneumonia\n\nOverview:\nCommunity-acquired pneumonia (CAP) is an acute lower respiratory tract infection associated with new radiographic shadowing not due to other causes, in patients not recently hospitalised.\n\nClinical presentation:\nSuspect pneumonia in patients with cough, sputum, dyspnoea, fever, pleuritic chest pain, or systemic illness, especially in older adults.\n\nSeverity assessment:\nUse CRB-65 in primary care or CURB-65 in hospital:\n- Confusion (new disorientation in time, place, or person)\n- Urea above 7 mmol/L (CURB only)\n- Respiratory rate 30/min or above\n- Blood pressure systolic below 90 or diastolic 60 or below\n- Age 65 or above\n\nScore interpretation (CURB-65):\n- 0-1: low severity, consider home treatment\n- 2: moderate severity, consider hospital assessment\n- 3-5: severe, hospital admission, consider intensive care\n\nInvestigations in hospital:\n- chest X-ray\n- FBC, U&E, CRP, LFTs\n- blood and sputum cultures\n- pulse oximetry; ABG if SpO2 below 94% or severe disease\n- atypical pathogen testing in moderate or severe CAP\n- consider urinary antigen tests for Legionella and pneumococcus\n\nAntibiotic treatment in CAP:\n- Low severity (CRB-65 0): oral amoxicillin 500 mg three times daily for 5 days\n- Moderate severity: dual oral therapy with amoxicillin and a macrolide for 5 days\n- High severity: IV co-amoxiclav and macrolide; consider levofloxacin in beta-lactam allergy\n\nDuration:\nRe-evaluate at 5 days. Most adults can stop treatment if clinically stable. Extend to 7-10 days for high-severity disease, atypical or staphylococcal pneumonia, or slow response.\n\nHospital-acquired pneumonia:\nSuspect HAP in patients with new symptoms 48 hours or more after hospital admission. Use local antibiotic protocols, considering broader spectrum to cover gram-negative organisms.\n\nDischarge criteria:\nAvoid discharge in patients who have two or more of: temperature above 37.5C, respiratory rate 24/min or above, heart rate above 100/min, systolic BP 90 or below, SpO2 below 90% on room air, abnormal mental status, inability to eat without assistance.\n"
  },
  {
    "filename": "acs.txt",
    "folder_key": "raw_nice",
    "content": "Title: Acute Coronary Syndromes\nSource: NICE\nMode: professional\nTopic: Acute coronary syndromes\n\nOverview:\nAcute coronary syndromes (ACS) include unstable angina, non-ST-elevation myocardial infarction (NSTEMI), and ST-elevation myocardial infarction (STEMI). They are caused by acute reduction in coronary blood flow, usually due to plaque rupture and thrombosis.\n\nInitial assessment:\nSuspect ACS in patients with chest pain or symptoms suggestive of cardiac ischaemia. Perform 12-lead ECG within 10 minutes of first medical contact.\n\nECG findings:\n- STEMI: ST-elevation 1 mm or more in 2 contiguous limb leads, 2 mm or more in 2 contiguous chest leads, or new left bundle branch block\n- NSTEMI/unstable angina: ST-depression, T-wave inversion, or non-specific changes; troponin distinguishes\n\nBiomarkers:\nMeasure high-sensitivity cardiac troponin at presentation and at 3 hours. A rising or falling pattern with at least one value above the 99th percentile defines myocardial infarction.\n\nInitial management of all suspected ACS:\n- aspirin 300 mg orally\n- glyceryl trinitrate sublingual or buccal\n- pain relief with morphine if severe\n- oxygen if SpO2 below 94%\n\nSTEMI management:\nOffer primary PCI to people with STEMI presenting within 12 hours of symptom onset, if PCI can be delivered within 120 minutes of when fibrinolysis could have been given. Otherwise offer fibrinolysis.\n\nAntiplatelet therapy in STEMI:\nAspirin 300 mg loading then 75 mg daily, plus a P2Y12 inhibitor (prasugrel preferred if going for PCI in patients without high bleeding risk; ticagrelor or clopidogrel are alternatives).\n\nNSTEMI/unstable angina management:\nRisk-stratify using GRACE score. Offer coronary angiography (with follow-on PCI if indicated) within 72 hours to patients with GRACE intermediate or higher risk; within 24 hours for very high-risk features.\n\nAntiplatelet therapy in NSTEMI:\nAspirin 300 mg loading then 75 mg daily plus a P2Y12 inhibitor for 12 months. Add fondaparinux unless going for immediate angiography.\n\nSecondary prevention:\n- dual antiplatelet therapy for 12 months (aspirin lifelong)\n- ACE inhibitor (or ARB)\n- beta-blocker\n- high-intensity statin (atorvastatin 80 mg)\n- cardiac rehabilitation\n- lifestyle advice\n\nCardiac rehabilitation:\nOffer to all patients post-MI. Should include exercise component, education, stress management, and risk-factor modification.\n"
  }
]

written = 0
for doc in CORPUS_PAYLOAD:
    target = f"{PATHS[doc['folder_key']]}/{doc['filename']}"
    with open(target, "w", encoding="utf-8") as f:
        f.write(doc["content"])
    written += 1

print(f"Wrote {written} corpus documents.")
print(f"  NHS  (patient mode):       {len(os.listdir(PATHS['raw_nhs']))}")
print(f"  NICE (professional mode):  {len(os.listdir(PATHS['raw_nice']))}")

Wrote 30 corpus documents.
  NHS  (patient mode):       15
  NICE (professional mode):  15



## 3. Chunking:

Boundary-aware splitter producing ~700-character chunks with 120-character
overlap. Prefers paragraph breaks, then newlines, then sentence ends, so
clinical meaning is preserved across chunk borders.

Each chunk inherits the parent document's metadata (`title`, `source`,
`mode`, `topic`), which enables the mode filter at retrieval time.

**Output:** `chunks.jsonl` on Drive, one JSON object per line.

In [ ]:
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120


def parse_doc(content: str, file_path: str) -> dict:
    """Extract metadata header (Title/Source/Mode/Topic) and body."""
    metadata = {"title": "", "source": "", "mode": "", "topic": ""}
    body_lines = []
    in_header = True
    for line in content.splitlines():
        s = line.strip()
        if in_header:
            # Header lines look like "Field: value". Anything else closes
            # the header and the rest is body.
            for key in metadata:
                prefix = f"{key.title()}:"
                if s.startswith(prefix):
                    metadata[key] = s.replace(prefix, "", 1).strip().lower() if key == "mode" else s.replace(prefix, "", 1).strip()
                    break
            else:
                if s == "":
                    continue
                in_header = False
                body_lines.append(line)
        else:
            body_lines.append(line)
    # Fallbacks if any field is empty
    if not metadata["title"]:
        metadata["title"] = Path(file_path).stem.replace("_", " ").title()
    if not metadata["source"]:  metadata["source"] = "UNKNOWN"
    if not metadata["mode"]:    metadata["mode"]   = "patient"
    if not metadata["topic"]:   metadata["topic"]  = metadata["title"]

    return {
        "doc_id": str(uuid.uuid4()),
        "file_path": file_path,
        **metadata,
        "text": "\n".join(body_lines).strip(),
    }


def boundary_aware_split(text: str, size: int, overlap: int) -> list:
    """Split `text` into ~size chunks, preferring natural boundaries.
    The overlap stops a single sentence being split across chunk borders
    (which would break clinical meaning) — empirically 120 chars is
    enough to bridge a typical sentence."""
    text = text.strip()
    if len(text) <= size:
        return [text] if text else []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        candidate = text[start:end]

        # Only adjust the boundary if we're not at the end of the doc
        if end < len(text):
            for sep in ["\n\n", "\n", ". "]:
                idx = candidate.rfind(sep)
                # Only accept the boundary if it's at least halfway through
                # — otherwise we end up with tiny chunks
                if idx > size * 0.5:
                    end = start + idx + (len(sep) - 1)
                    break

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


# Build chunks from every .txt file under data/raw/
all_chunks = []
for folder in [PATHS["raw_nhs"], PATHS["raw_nice"]]:
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith(".txt"):
            continue
        path = os.path.join(folder, fn)
        doc = parse_doc(Path(path).read_text(encoding="utf-8"), path)
        for i, txt in enumerate(boundary_aware_split(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
            all_chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "doc_id": doc["doc_id"],
                "chunk_index": i,
                "title": doc["title"],
                "source": doc["source"],
                "mode": doc["mode"],
                "topic": doc["topic"],
                "file_path": doc["file_path"],
                "text": txt,
            })

with open(CHUNK_FILE, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# Summary so we can sanity-check before embedding
patient_n = sum(1 for c in all_chunks if c["mode"] == "patient")
prof_n = sum(1 for c in all_chunks if c["mode"] == "professional")
print(f"Chunks total:   {len(all_chunks)}")
print(f"  patient:      {patient_n}")
print(f"  professional: {prof_n}")
print(f"Saved -> {CHUNK_FILE}")

Chunks total:   118
  patient:      51
  professional: 67
Saved -> /content/drive/MyDrive/educare_ai/data/processed/chunks.jsonl



## 4. Embeddings and FAISS indices:

`sentence-transformers/all-MiniLM-L6-v2` &mdash; 384-dim, L2-normalised, so
inner-product search on FAISS is cosine similarity.

We build **two** indices:

- **HNSWFlat** (M=32, efSearch=64) &mdash; production retriever, fast at scale.
- **IndexFlatIP** &mdash; exact baseline kept for evaluation reproducibility.

Both persist to Drive and subsequent runs load from disk in milliseconds.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Load chunks back from disk so this section is independent
chunks = [json.loads(l) for l in open(CHUNK_FILE) if l.strip()]
print(f"Loaded {len(chunks)} chunks")

# MiniLM is small enough to embed the whole corpus on CPU in seconds.
# Larger models would offer marginal gains but break the "runs anywhere"
# constraint we deliberately keep for marker reproducibility.
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(
    texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True,
).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

Loaded 118 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (118, 384)


In [ ]:
dim = embeddings.shape[1]

# Production index: HNSW. Sub-millisecond retrieval that scales to
# thousands of chunks. M=32 keeps memory reasonable; efSearch=64 holds
# >99% recall vs the exact baseline at our scale (verified in §11).
hnsw = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.hnsw.efSearch = 64
hnsw.add(embeddings)
faiss.write_index(hnsw, HNSW_PATH)

# Eval-baseline index: exact cosine. Used by §11's recall test to prove
# HNSW is safe for production.
exact = faiss.IndexFlatIP(dim)
exact.add(embeddings)
faiss.write_index(exact, EXACT_PATH)

# Persist chunk metadata alongside the index so retrieval can map
# vector indices back to {title, source, mode, topic, text}.
with open(META_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# Active retriever is HNSW; `exact` stays loaded for §11.
index = hnsw
print(f"HNSW  index : {hnsw.ntotal} vectors -> {HNSW_PATH}")
print(f"Exact index : {exact.ntotal} vectors -> {EXACT_PATH}")
print(f"Active      : HNSW (M=32, efSearch=64)")

HNSW  index : 118 vectors -> /content/drive/MyDrive/educare_ai/data/index/faiss_hnsw.index
Exact index : 118 vectors -> /content/drive/MyDrive/educare_ai/data/index/faiss_exact.index
Active      : HNSW (M=32, efSearch=64)


## 5. Retrieval:

Mode-aware top-k with content-hash deduplication.

**Why over-fetch + filter:** if we asked FAISS for top-3 directly and then
filtered, a query that lands hot on professional chunks could starve the
patient mode entirely. Over-fetching `top_k * 4` then filtering preserves
result quality across both modes.

In [ ]:
def retrieve_chunks(query: str, mode: str = "patient", top_k: int = 3) -> list:
    """Return top_k chunks matching mode. Always over-fetches first."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    raw_k = min(max(top_k * 4, 10), index.ntotal)
    scores, idxs = index.search(q_emb, raw_k)

    results = []
    seen_hashes = set()
    for score, i in zip(scores[0], idxs[0]):
        if i < 0:
            continue
        c = chunks[i]
        if c["mode"] != mode:
            continue
        # Content-hash dedup catches near-duplicate chunks that ended up
        # in the top-k together (happens with overlapping chunk windows)
        h = hashlib.md5(c["text"].encode()).hexdigest()
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        results.append({
            "chunk_id": c["chunk_id"],
            "title": c["title"], "source": c["source"], "mode": c["mode"],
            "topic": c["topic"], "text": c["text"], "score": float(score),
        })
        if len(results) >= top_k:
            break
    return results


# Smoke test before moving on
test = retrieve_chunks("What are the symptoms of asthma?", mode="patient", top_k=3)
print(f"Retrieved {len(test)} chunks for the test query")
for r in test:
    print(f"  [{r['source']}] {r['title']} (score={r['score']:.3f})")

Retrieved 3 chunks for the test query
  [NHS] Asthma (score=0.757)
  [NHS] Asthma (score=0.587)
  [NHS] Allergies (score=0.502)


---
## 6. Generation &mdash; flan-t5-base

The model that turns retrieved chunks into a mode-appropriate answer.

**Why flan-t5-base.** Sprint 2 used flan-t5-base (250M). It's reliable, runs comfortably on a Colab T4 GPU (or CPU), and serves as a fast baseline, though it can be a bit more terse than the larger variants.

**Why not a hosted API.** Reproducibility. A marker should be able to run
this notebook without an API key.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# fp16 cuts memory roughly in half and inference time by ~30% on a T4.
# CPU fallback uses fp32 because half-precision on CPU is slower than full.
GEN_MODEL_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {GEN_MODEL_NAME} on {device} ({dtype}) ...")
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(
    GEN_MODEL_NAME, torch_dtype=dtype,
).to(device).eval()
print(f"Generator ready on {device}")

Loading google/flan-t5-base on cuda (torch.float16) ...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready on cuda


### 6.1 Mode-specific prompts:

Patient mode demands plain English with bracketed explanations of any
medical term; this is what produces the readability gain. Professional
mode demands clinical terminology and exact thresholds.

Both prompts include the `[Source N]` citation-marker instruction so the
LLM emits inline attributions itself, reducing reliance on the
post-processor.

In [ ]:
PATIENT_PROMPT = """You are a friendly health information assistant for patients.
Use ONLY the context below to answer the question. Do not invent facts.
If the context does not contain the answer, say you do not have enough information.

WRITING STYLE:
- Plain everyday English. Short sentences.
- Avoid medical jargon. If a medical term cannot be avoided, immediately
  explain it in brackets in simpler words.
- Cover all main points from the context. Be warm and reassuring.
- Do NOT give specific medication doses unless they are clearly stated
  in the context.

CITATIONS:
- After each sentence that uses a numbered source, add [Source N]
  immediately after that sentence.

Context:
{context}

Question: {question}

Answer in plain English (with [Source N] citations):"""


PROFESSIONAL_PROMPT = """You are a clinical reference assistant for healthcare professionals and students.
Use ONLY the context below. Do not invent facts.
If the context does not contain the answer, say the guideline does not specify.

WRITING STYLE:
- Concise, structured, guideline-style.
- Use clinical terminology where appropriate.
- Preserve thresholds, doses, and staging exactly as written
  (e.g. "140/90 mmHg", "HbA1c 48 mmol/mol", "ABPM 135/85").
- Cover all relevant points from the context.

CITATIONS:
- After each clinical statement that uses a numbered source, add
  [Source N] immediately after that sentence.

Context:
{context}

Question: {question}

Answer (clinical style, with [Source N] citations):"""


def build_prompt(query: str, retrieved: list, mode: str) -> str:
    parts = [
        f"[Source {i}: {r['source']} - {r['title']}]\n{r['text']}"
        for i, r in enumerate(retrieved, start=1)
    ]
    template = PATIENT_PROMPT if mode == "patient" else PROFESSIONAL_PROMPT
    return template.format(context="\n\n".join(parts), question=query)

### 6.2 Generation function :

Beam search with `length_penalty=1.4` mildly biases toward more complete
answer, flan-t5-large still tends toward the shortest generation that
satisfies the prompt, which loses keywords the eval harness checks for.

In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 512) -> str:
    """Run the LLM and decode. max_new_tokens=512 gives the model room
    to elaborate; truncated to 1024 input tokens to fit T5's window
    along with the generation budget."""
    inputs = gen_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1024,
    ).to(device)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
            length_penalty=1.4,
        )
    return gen_tokenizer.decode(out[0], skip_special_tokens=True).strip()


## 7. Safety layer:

Two guards run before the LLM:

1. **Categorised emergency classifier.** Six categories, each with a
   regex pattern set covering common clinical and lay phrasings. Returns
   `(flag, category)` so the UI can show category-specific copy.
2. **Patient-mode dose refusal.** Short-circuits prescription-dose
   questions before they reach the LLM, routing the user to a clinician.

The pattern set is deliberately broad: emergency *recall* matters more
than precision &mdash; a false positive shows an extra warning, a false
negative misses a real distress signal.

In [ ]:
# Pattern set tuned for high recall on common phrasings. Word boundaries
# (\b) prevent spurious matches; alternatives (?:...) cover lay variants.
EMERGENCY_PATTERNS = {
    "cardiac": [
        r"\bchest\s+pain\b", r"\bcrushing\b", r"\bheart\s+attack\b",
        r"\bmyocardial\s+infarction\b", r"\bMI\b",
        r"\bsevere\s+chest\b", r"\btightness\s+in\s+(?:my\s+)?chest\b",
        r"\bchest\s+(?:is\s+)?crushing\b", r"\bpain\s+in\s+(?:my\s+)?chest\b",
    ],
    "respiratory": [
        r"\bcan(?:no|')?t\s+breathe\b", r"\bcannot\s+breathe\b",
        r"\b(?:severe|bad)\s+(?:shortness\s+of\s+breath|breathlessness)\b",
        r"\basthma\s+attack\b", r"\bchoking\b",
        r"\bgasping\s+for\s+(?:air|breath)\b",
        r"\bstruggling\s+to\s+breathe\b", r"\bunable\s+to\s+breathe\b",
        r"\bnot\s+breathing\b",
    ],
    "neurological": [
        r"\bstroke\b", r"\bface\s+(?:dropping|drooping|fallen)\b",
        r"\bslurred\s+speech\b", r"\bcan(?:no|')?t\s+speak\b",
        r"\bone[- ]?sided\s+weakness\b", r"\b(?:arm|leg)\s+(?:gone\s+)?numb\b",
        r"\bseizure\b", r"\bunconscious\b", r"\bcollaps(?:ed|ing)\b",
        r"\bsudden\s+(?:severe\s+)?headache\b",
        r"\bworst\s+headache\s+(?:of\s+my\s+life|ever)\b",
    ],
    "anaphylaxis": [
        r"\banaphylaxis\b", r"\banaphylactic\b",
        r"\bswollen\s+(?:tongue|throat|lips|face)\b",
        r"\bcan(?:no|')?t\s+swallow\b",
        r"\bthroat\s+(?:closing|tightening|swelling)\b",
        r"\bsevere\s+allergic\s+reaction\b",
    ],
    "paediatric": [
        r"\bbaby\s+(?:not|won'?t|isn'?t)\s+breathing\b",
        r"\bblue\s+(?:lips|baby|child)\b",
        r"\bunresponsive\s+(?:baby|child|infant)\b",
        r"\b(?:baby|child|infant)\s+(?:limp|floppy|unresponsive)\b",
    ],
    "self_harm": [
        r"\boverdose\b", r"\boverdosed\b", r"\btoo\s+many\s+pills\b",
        r"\bsuicid(?:e|al|ing)\b", r"\bkill\s+myself\b",
        r"\bharm\s+myself\b", r"\bend\s+(?:my\s+)?(?:life|it\s+all)\b",
        r"\bdon'?t\s+want\s+to\s+(?:live|be\s+here|be\s+alive)\b",
        r"\btaking\s+(?:my\s+)?own\s+life\b",
    ],
}


def detect_emergency(query: str):
    """Return (flag, category) — first matching category wins."""
    q = query.lower()
    for category, patterns in EMERGENCY_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, q):
                return True, category
    return False, None


# Patient-mode prescription-dose refusal. Matches any phrasing that
# implies "tell me how much medication to take".
DOSE_QUESTION = re.compile(
    r"\b(?:how\s+much|what\s+dose|dosage|how\s+many\s+(?:mg|tablets|pills|doses)|"
    r"mg\s+(?:of|should)|milligrams?|safe\s+(?:dose|amount))\b",
    re.IGNORECASE,
)

def patient_dose_refusal(query: str) -> bool:
    return bool(DOSE_QUESTION.search(query))


SAFETY_NOTE = (
    "This information is for educational purposes only and does not "
    "replace professional medical advice."
)
EMERGENCY_MESSAGE = (
    "If this may be a medical emergency, seek urgent medical help "
    "immediately or call emergency services (999 in the UK, 911 in the US)."
)
DOSE_REFUSAL_MESSAGE = (
    "For your safety, EduCare AI does not provide specific medication doses "
    "in Patient mode. Please follow the dose written on your prescription "
    "label, or speak to your GP, pharmacist, or NHS 111."
)

# Quick sanity check
for q in ["What are asthma symptoms?", "I cant breathe", "How much paracetamol can I take?",
          "I want to end my life", "My baby is limp"]:
    flag, cat = detect_emergency(q)
    dose = patient_dose_refusal(q)
    print(f"  {q!r:<50} emergency={flag}, category={cat}, dose_refusal={dose}")

  'What are asthma symptoms?'                        emergency=False, category=None, dose_refusal=False
  'I cant breathe'                                   emergency=True, category=respiratory, dose_refusal=False
  'How much paracetamol can I take?'                 emergency=False, category=None, dose_refusal=True
  'I want to end my life'                            emergency=True, category=self_harm, dose_refusal=False
  'My baby is limp'                                  emergency=False, category=None, dose_refusal=False



## 8. End-to-end answer pipeline:

`educare_answer(query, mode, top_k)` is the single entry point used by
the API, the demo UI, and the evaluation harness. Returns a dict matching
the React front-end's expected shape:

```
{
  "query", "mode",
  "answer_text", "citations": [{"title", "source", "url"}],
  "emergency_flag", "emergency_category",
  "safety_note", "refusal"
}
```

In [ ]:
# NHS condition pages follow a predictable URL scheme; NICE guidelines
# don't, so we link to the index page and let the title disambiguate.
def build_citation(c: dict) -> dict:
    src, title = c["source"], c["title"]
    if src == "NHS":
        slug = title.lower().replace(" (", "-").replace(")", "") \
                    .replace(" ", "-").replace("/", "-").replace("--", "-").strip("-")
        url = f"https://www.nhs.uk/conditions/{slug}/"
    elif src == "NICE":
        url = "https://www.nice.org.uk/guidance"
    else:
        url = "#"
    return {"title": title, "source": src, "url": url}


def educare_answer(query: str, mode: str = "patient", top_k: int = 3) -> dict:
    emergency_flag, emergency_category = detect_emergency(query)

    # Patient-mode dose refusal: short-circuit before retrieval, so the
    # LLM never sees the question at all.
    if mode == "patient" and patient_dose_refusal(query):
        return {
            "query": query, "mode": mode,
            "answer_text": DOSE_REFUSAL_MESSAGE,
            "citations": [],
            "emergency_flag": emergency_flag,
            "emergency_category": emergency_category,
            "safety_note": SAFETY_NOTE, "refusal": True,
        }

    retrieved = retrieve_chunks(query, mode=mode, top_k=top_k)
    if not retrieved:
        return {
            "query": query, "mode": mode,
            "answer_text": (
                "I do not have enough information in the EduCare AI corpus "
                "to answer this question. Please consult NHS 111, your GP, "
                "or a trusted clinical reference."
            ),
            "citations": [],
            "emergency_flag": emergency_flag,
            "emergency_category": emergency_category,
            "safety_note": SAFETY_NOTE, "refusal": False,
        }

    prompt = build_prompt(query, retrieved, mode=mode)
    generated = generate_answer(prompt)

    # Dedup citations on (source, title) so the same NHS page only appears
    # once even if multiple chunks from it were retrieved
    seen, citations = set(), []
    for r in retrieved:
        key = (r["source"], r["title"])
        if key in seen:
            continue
        seen.add(key)
        citations.append(build_citation(r))

    # Fallback: if the LLM didn't insert [Source N] markers, append a
    # citation list. The post-processor is the safety net, not the
    # primary attribution mechanism.
    answer_text = generated
    if not any(f"[Source {i}]" in answer_text for i in range(1, len(retrieved) + 1)):
        cites = "; ".join(f"{c['source']} \u2014 {c['title']}" for c in citations)
        answer_text += f"\n\n[Sources: {cites}]"

    if emergency_flag:
        answer_text = f"\u26a0\ufe0f URGENT SAFETY ADVICE\n{EMERGENCY_MESSAGE}\n\n{answer_text}"

    return {
        "query": query, "mode": mode,
        "answer_text": answer_text, "citations": citations,
        "emergency_flag": emergency_flag,
        "emergency_category": emergency_category,
        "safety_note": SAFETY_NOTE, "refusal": False,
    }

### 8.1 Demo (same question, two modes):

Confirms the dual-mode deliverable works before standing up the API.

In [ ]:
import textwrap

def show(r):
    print(f"--- mode={r['mode']}  emergency={r['emergency_flag']} ({r['emergency_category']})  refusal={r['refusal']} ---")
    print(textwrap.fill(r["answer_text"], width=88))
    print("\nCitations:")
    for c in r["citations"]:
        print(f"  - {c['source']} \u2014 {c['title']} ({c['url']})")
    print()

show(educare_answer("What are the symptoms of asthma?", mode="patient"))
show(educare_answer("How is hypertension diagnosed?", mode="professional"))
show(educare_answer("I'm having an asthma attack and can't breathe", mode="patient"))
show(educare_answer("How much paracetamol can my child take?", mode="patient"))

--- mode=patient  emergency=False (None)  refusal=False ---
wheezing, coughing, shortness of breath, and chest tight  [Sources: NHS — Asthma; NHS —
Allergies]

Citations:
  - NHS — Asthma (https://www.nhs.uk/conditions/asthma/)
  - NHS — Allergies (https://www.nhs.uk/conditions/allergies/)

--- mode=professional  emergency=False (None)  refusal=False ---
Suspect hypertension if clinic systolic blood pressure is 140 mmHg  [Sources: NICE —
Hypertension]

Citations:
  - NICE — Hypertension (https://www.nice.org.uk/guidance)

--- mode=patient  emergency=True (respiratory)  refusal=False ---
⚠️ URGENT SAFETY ADVICE If this may be a medical emergency, seek urgent medical help
immediately or call emergency services (999 in the UK, 911 in the US).  Do NOT give
specific medication doses unless they are clearly stated in the context  [Sources: NHS —
Asthma]

Citations:
  - NHS — Asthma (https://www.nhs.uk/conditions/asthma/)

--- mode=patient  emergency=False (None)  refusal=True ---
For your sa

---
## 9. FastAPI service :

Endpoints:

| Endpoint | Purpose | Rate limit |
|---|---|---|
| `GET /health` | Service readiness | none |
| `POST /retrieve` | Top-k chunks (no generation) | 120/min/IP |
| `POST /answer` | Full RAG pipeline (production traffic) | 60/min/IP |
| `POST /answer_eval` | Identical to `/answer`, rate-limit-exempt | none |
| `POST /sessions` | Open a structured user-testing session | none |
| `POST /feedback` | Submit a per-question feedback record | none |
| `GET /feedback/stats` | Lightweight monitoring | none |

Two design decisions worth noting:

- **`/answer_eval` exists** because the production rate limit on `/answer`
  correctly rejects burst tests as 429s. We need the limit on production,
  but not in evaluation &mdash; otherwise the load test measures the rate
  limiter, not the system.
- **CORS open to `*`** for the academic demo. Sprint 4 would tighten to
  the deployed front-end origin.

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Literal, List, Optional
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded


# Pydantic contracts. Constrained types catch malformed requests with a
# 422 before they reach the model.
ModeType = Literal["patient", "professional"]

class AnswerRequest(BaseModel):
    query: str = Field(..., min_length=2, max_length=1000)
    mode: ModeType = "patient"
    top_k: int = Field(default=3, ge=1, le=8)

class Citation(BaseModel):
    title: str; source: str; url: str

class AnswerResponse(BaseModel):
    query: str; mode: ModeType
    answer_text: str
    citations: List[Citation]
    emergency_flag: bool
    emergency_category: Optional[str] = None
    safety_note: str
    refusal: bool

class RetrieveRequest(BaseModel):
    query: str = Field(..., min_length=2, max_length=1000)
    mode: ModeType = "patient"
    top_k: int = Field(default=3, ge=1, le=8)

class ChunkResult(BaseModel):
    chunk_id: str; title: str; source: str; mode: str
    topic: str; text: str; score: float

class RetrieveResponse(BaseModel):
    query: str; mode: ModeType; results: List[ChunkResult]


# User-testing models. Sessions and feedback persist as JSONL on Drive
# so Section 12 can ingest them.
class SessionStartRequest(BaseModel):
    role: ModeType
    participant_alias: Optional[str] = None

class SessionStartResponse(BaseModel):
    session_id: str
    started_at: float

class FeedbackRequest(BaseModel):
    session_id: str
    query: str
    mode: ModeType
    answer_text: str
    accuracy_rating: int = Field(..., ge=1, le=5)
    clarity_rating: int = Field(..., ge=1, le=5)
    trust_rating: int = Field(..., ge=1, le=5)
    citations_helpful: bool
    comment: Optional[str] = ""

class FeedbackResponse(BaseModel):
    feedback_id: str
    received_at: float


app = FastAPI(title="EduCare AI API", version="3.1.0-final")

# CORS: open during the academic demo. Tighten in production.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"], allow_headers=["*"],
)

# Rate limiting: 60/min/IP on /answer protects the LLM from runaway clients.
# /answer_eval skips this so the evaluation harness isn't throttled.
RATE_LIMIT = os.environ.get("EDUCARE_RATE_LIMIT", "60/minute")
limiter = Limiter(key_func=get_remote_address, default_limits=[RATE_LIMIT])
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)


@app.get("/health")
def health():
    return {"status": "ok", "version": "3.1.0-final",
            "vectors": int(index.ntotal),
            "documents": len({c["doc_id"] for c in chunks})}

@app.post("/retrieve", response_model=RetrieveResponse)
@limiter.limit("120/minute")
def api_retrieve(request: Request, payload: RetrieveRequest):
    try:
        return {"query": payload.query, "mode": payload.mode,
                "results": retrieve_chunks(payload.query, payload.mode, payload.top_k)}
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/answer", response_model=AnswerResponse)
@limiter.limit("60/minute")
def api_answer(request: Request, payload: AnswerRequest):
    try:
        return educare_answer(payload.query, payload.mode, payload.top_k)
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/answer_eval", response_model=AnswerResponse)
def api_answer_eval(payload: AnswerRequest):
    """Rate-limit-exempt version for the evaluation harness only."""
    try:
        return educare_answer(payload.query, payload.mode, payload.top_k)
    except Exception as e:
        raise HTTPException(500, str(e))


@app.post("/sessions", response_model=SessionStartResponse)
def start_session(payload: SessionStartRequest):
    sid = f"sess_{uuid.uuid4().hex[:12]}"
    rec = {"session_id": sid, "role": payload.role,
           "participant_alias": payload.participant_alias or "anonymous",
           "started_at": time.time()}
    with open(SESSIONS_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return {"session_id": sid, "started_at": rec["started_at"]}

@app.post("/feedback", response_model=FeedbackResponse)
def submit_feedback(payload: FeedbackRequest):
    fid = f"fb_{uuid.uuid4().hex[:12]}"
    rec = {"feedback_id": fid, **payload.model_dump(), "received_at": time.time()}
    with open(FEEDBACK_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return {"feedback_id": fid, "received_at": rec["received_at"]}

@app.get("/feedback/stats")
def feedback_stats():
    if not os.path.exists(FEEDBACK_LOG):
        return {"total": 0}
    with open(FEEDBACK_LOG) as f:
        n = sum(1 for line in f if line.strip())
    return {"total": n}


print("API defined: /health /retrieve /answer /answer_eval /sessions /feedback /feedback/stats")

API defined: /health /retrieve /answer /answer_eval /sessions /feedback /feedback/stats



## 10. Demo (Gradio UI and public API):

Runs Uvicorn in a background thread, exposes via ngrok, launches the
Gradio in-notebook UI for live demos and viva.

In [ ]:
import nest_asyncio, uvicorn
from threading import Thread

nest_asyncio.apply()

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

Thread(target=_serve, daemon=True).start()
time.sleep(3)
print("FastAPI running on :8000")

FastAPI running on :8000


In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import getpass

# Token from Colab Secrets if available, otherwise prompt once
try:
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = getpass.getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(token)

# Disconnect any stale tunnels from earlier runs
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8000).public_url
print(f"Public URL: {public_url}")
print(f"Swagger:    {public_url}/docs")

Paste your ngrok authtoken: ··········
Public URL: https://df50-34-16-240-226.ngrok-free.app
Swagger:    https://df50-34-16-240-226.ngrok-free.app/docs


In [ ]:
import gradio as gr
import requests

def call_api(q, mode, k):
    if not q.strip():
        return "Enter a question.", "", "No", "—"
    try:
        r = requests.post(f"{public_url}/answer",
                          json={"query": q.strip(), "mode": mode, "top_k": int(k)},
                          timeout=120)
        r.raise_for_status()
        d = r.json()
        cites = "\n".join(f"- {c['source']} \u2014 {c['title']}" for c in d.get("citations", [])) or "—"
        return d["answer_text"], cites, "Yes" if d["emergency_flag"] else "No", d.get("emergency_category") or "—"
    except Exception as e:
        return f"API error: {e}", "—", "No", "—"

with gr.Blocks(title="EduCare AI") as demo:
    gr.Markdown("# EduCare AI &mdash; Sprint 3 Final")
    q = gr.Textbox(label="Question", lines=3, placeholder="Ask a healthcare question…")
    with gr.Row():
        mode = gr.Radio(["patient", "professional"], value="patient", label="Mode")
        k = gr.Slider(1, 5, value=3, step=1, label="Top K")
    submit = gr.Button("Get Answer", variant="primary")
    answer = gr.Textbox(label="Answer", lines=12)
    with gr.Row():
        cites = gr.Textbox(label="Sources", lines=4)
        with gr.Column():
            emerg = gr.Textbox(label="Emergency", lines=1)
            cat = gr.Textbox(label="Category", lines=1)

    gr.Examples(
        [
            ["What are the symptoms of asthma?", "patient", 3],
            ["I'm having an asthma attack and can't breathe", "patient", 3],
            ["How much paracetamol can my child take?", "patient", 3],
            ["How is hypertension diagnosed?", "professional", 3],
            ["First-line drug for type 2 diabetes?", "professional", 3],
            ["High-risk criteria for sepsis?", "professional", 3],
        ],
        inputs=[q, mode, k],
    )
    submit.click(call_api, inputs=[q, mode, k], outputs=[answer, cites, emerg, cat])

demo.launch(debug=False, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d57c85b6dd2e972920.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



## 11. Evaluation harness:

36-question gold bank covering all 30 corpus documents (15 patient + 15
professional + 3 emergency triggers + 3 mode-routing traps), plus:

- Citation presence and provenance
- Flesch&ndash;Kincaid readability gain (patient mode)
- Sequential and concurrent latency
- Safety recall and precision
- Mode-routing correctness
- HNSW vs exact-index recall

All metrics from the proposal are measured here. Outputs land in
`data/evaluation/`.

### 11.1 Stem-aware keyword matcher

A medical answer that says "wheezing, coughing, breathlessness" should
satisfy a gold list of `["wheeze", "cough", "breath"]` &mdash; the meaning
is identical, the morphology differs. The matcher accepts common English
morphological variants but enforces word boundaries, so "rest" doesn't
spuriously match inside "arrested".

In [ ]:
def _morph_variants(k: str) -> list:
    """Return the keyword plus common -s, -ed, -ing, -less, -ly variants."""
    k = k.lower().strip()
    if " " in k or len(k) < 3:
        return [k]
    v = {k, k + "s"}
    if k.endswith("y"):    v.add(k[:-1] + "ies")
    if k.endswith("e"):
        v.update([k + "d", k[:-1] + "ing"])
    else:
        v.update([k + "ed", k + "ing"])
    v.update([k + "less", k + "lessness", k + "y", k + "ly"])
    return list(v)

def _kw_in(text_lc: str, kw: str) -> bool:
    """Check if any morphological variant of kw appears as a whole word."""
    return any(re.search(r"\b" + re.escape(v) + r"\b", text_lc)
               for v in _morph_variants(kw))

def keyword_match(text: str, keywords: list) -> bool:
    if not keywords: return True
    t = text.lower()
    return all(_kw_in(t, k) for k in keywords)

def keyword_absent(text: str, keywords: list) -> bool:
    if not keywords: return True
    t = text.lower()
    return not any(_kw_in(t, k) for k in keywords)

# Self-check that the matcher behaves correctly
assert keyword_match("Patients have wheezing and dry cough.", ["wheeze", "cough"])
assert keyword_match("Severe breathlessness occurs.", ["breath"])
assert not keyword_match("Patient was arrested.", ["rest"])
print("Keyword matcher OK")

Keyword matcher OK


### 11.2 Gold-standard question bank (36 items)

In [ ]:
GOLD_BANK = [
    # Patient mode — 15 NHS conditions
    {"id": "P_AST_01", "mode": "patient", "query": "What are the symptoms of asthma?",
     "expected_topic": "Asthma", "expected_keywords": ["wheeze", "cough", "breath"]},
    {"id": "P_DIA_01", "mode": "patient", "query": "What are the symptoms of diabetes?",
     "expected_topic": "Diabetes", "expected_keywords": ["thirst", "tired"]},
    {"id": "P_HBP_01", "mode": "patient", "query": "Does high blood pressure cause symptoms?",
     "expected_topic": "High blood pressure", "expected_keywords": ["no symptoms"]},
    {"id": "P_STR_01", "mode": "patient", "query": "How can I recognise the signs of a stroke?",
     "expected_topic": "Stroke", "expected_keywords": ["FAST", "999"]},
    {"id": "P_HRT_01", "mode": "patient", "query": "What are the symptoms of a heart attack?",
     "expected_topic": "Heart attack", "expected_keywords": ["chest pain", "999"]},
    {"id": "P_CHL_01", "mode": "patient", "query": "How is high cholesterol diagnosed?",
     "expected_topic": "High cholesterol", "expected_keywords": ["blood test"]},
    {"id": "P_CPD_01", "mode": "patient", "query": "What are the symptoms of COPD?",
     "expected_topic": "COPD", "expected_keywords": ["breath", "cough"]},
    {"id": "P_DEP_01", "mode": "patient", "query": "What are the signs of depression?",
     "expected_topic": "Depression", "expected_keywords": ["sad"]},
    {"id": "P_ANX_01", "mode": "patient", "query": "What are the physical symptoms of anxiety?",
     "expected_topic": "Anxiety", "expected_keywords": ["palpitations"]},
    {"id": "P_FLU_01", "mode": "patient", "query": "What are the symptoms of flu?",
     "expected_topic": "Flu", "expected_keywords": ["temperature", "tired"]},
    {"id": "P_HDC_01", "mode": "patient", "query": "When should I worry about a headache?",
     "expected_topic": "Headache", "expected_keywords": ["999"]},
    {"id": "P_BCK_01", "mode": "patient", "query": "What can I do at home for back pain?",
     "expected_topic": "Back pain", "expected_keywords": ["active"]},
    {"id": "P_CLD_01", "mode": "patient", "query": "How do I treat a cold at home?",
     "expected_topic": "Common cold", "expected_keywords": ["rest"]},
    {"id": "P_UTI_01", "mode": "patient", "query": "What are the symptoms of a urinary tract infection?",
     "expected_topic": "Urinary tract infection", "expected_keywords": ["burning"]},
    {"id": "P_ALR_01", "mode": "patient", "query": "What is anaphylaxis?",
     "expected_topic": "Allergies", "expected_keywords": ["999"]},
    # Patient mode — 3 emergency triggers
    {"id": "P_EME_01", "mode": "patient", "query": "I'm having an asthma attack and can't breathe",
     "expected_topic": "Asthma", "expected_keywords": ["999"]},
    {"id": "P_EME_02", "mode": "patient", "query": "I think I'm having a stroke, my face is drooping",
     "expected_topic": "Stroke", "expected_keywords": []},
    {"id": "P_EME_03", "mode": "patient", "query": "I have severe chest pain",
     "expected_topic": "Heart attack", "expected_keywords": []},
    # Professional mode — 15 NICE topics
    {"id": "X_HYP_01", "mode": "professional", "query": "How is hypertension diagnosed?",
     "expected_topic": "Hypertension", "expected_keywords": ["140", "90", "ABPM"]},
    {"id": "X_STR_01", "mode": "professional", "query": "What should happen after a suspected TIA in the last week?",
     "expected_topic": "Stroke and TIA", "expected_keywords": ["aspirin", "24 hours"]},
    {"id": "X_AST_01", "mode": "professional", "query": "What is step 1 asthma therapy in adults?",
     "expected_topic": "Asthma chronic management", "expected_keywords": ["formoterol"]},
    {"id": "X_CPD_01", "mode": "professional", "query": "How is COPD severity classified by FEV1?",
     "expected_topic": "COPD", "expected_keywords": ["80", "50"]},
    {"id": "X_HF_01", "mode": "professional", "query": "What NT-proBNP threshold requires urgent specialist referral?",
     "expected_topic": "Heart failure", "expected_keywords": ["2000"]},
    {"id": "X_DEP_01", "mode": "professional", "query": "What is the first-line antidepressant for adults with depression?",
     "expected_topic": "Depression", "expected_keywords": ["SSRI"]},
    {"id": "X_GAD_01", "mode": "professional", "query": "What is the first-line drug treatment for generalised anxiety disorder?",
     "expected_topic": "Generalised anxiety disorder", "expected_keywords": ["sertraline"]},
    {"id": "X_T2D_01", "mode": "professional", "query": "What is the first-line drug for type 2 diabetes?",
     "expected_topic": "Type 2 diabetes", "expected_keywords": ["metformin"]},
    {"id": "X_T1D_01", "mode": "professional", "query": "What insulin regimen is preferred in adults with type 1 diabetes?",
     "expected_topic": "Type 1 diabetes", "expected_keywords": ["basal-bolus"]},
    {"id": "X_AF_01", "mode": "professional", "query": "What is the first-line anticoagulant for atrial fibrillation?",
     "expected_topic": "Atrial fibrillation", "expected_keywords": ["DOAC"]},
    {"id": "X_CKD_01", "mode": "professional", "query": "What eGFR defines CKD stage G3a?",
     "expected_topic": "Chronic kidney disease", "expected_keywords": ["45"]},
    {"id": "X_SEP_01", "mode": "professional", "query": "What are the high-risk criteria for suspected sepsis?",
     "expected_topic": "Sepsis", "expected_keywords": ["lactate"]},
    {"id": "X_AKI_01", "mode": "professional", "query": "What rise in serum creatinine defines acute kidney injury?",
     "expected_topic": "Acute kidney injury", "expected_keywords": ["26"]},
    {"id": "X_PNE_01", "mode": "professional", "query": "What is the CURB-65 score used for?",
     "expected_topic": "Pneumonia", "expected_keywords": ["severity"]},
    {"id": "X_ACS_01", "mode": "professional", "query": "What is the initial antiplatelet for suspected ACS?",
     "expected_topic": "Acute coronary syndromes", "expected_keywords": ["aspirin"]},
    # Professional — 3 mode-routing traps
    {"id": "X_TRP_01", "mode": "professional", "query": "What is the ABPM threshold for stage 1 hypertension?",
     "expected_topic": "Hypertension", "expected_keywords": ["135/85"]},
    {"id": "X_TRP_02", "mode": "professional", "query": "What is the target HbA1c on metformin monotherapy?",
     "expected_topic": "Type 2 diabetes", "expected_keywords": ["48"]},
    {"id": "X_TRP_03", "mode": "professional", "query": "What rehabilitation should follow stroke?",
     "expected_topic": "Stroke and TIA", "expected_keywords": ["multidisciplinary"]},
]
for q in GOLD_BANK:
    q.setdefault("forbidden_keywords", [])

print(f"Gold bank: {len(GOLD_BANK)} questions  "
      f"(patient={sum(1 for q in GOLD_BANK if q['mode']=='patient')}, "
      f"professional={sum(1 for q in GOLD_BANK if q['mode']=='professional')})")

Gold bank: 36 questions  (patient=18, professional=18)


### 11.3 Run accuracy + citation + latency benchmarks

In [ ]:
import pandas as pd
import numpy as np

def evaluate_item(item):
    t0 = time.perf_counter()
    result = educare_answer(item["query"], mode=item["mode"], top_k=3)
    elapsed = time.perf_counter() - t0
    retrieved = retrieve_chunks(item["query"], mode=item["mode"], top_k=3)

    topics = [r["topic"] for r in retrieved]
    modes = [r["mode"] for r in retrieved]
    answer = result["answer_text"]

    return {
        "id": item["id"], "mode": item["mode"], "query": item["query"],
        "expected_topic": item["expected_topic"],
        "retrieved_topics": ", ".join(set(topics)) if topics else "(none)",
        "topic_correct": item["expected_topic"] in topics if topics else False,
        "mode_correct": all(m == item["mode"] for m in modes) if modes else False,
        "keywords_present": keyword_match(answer, item["expected_keywords"]),
        "forbidden_absent": keyword_absent(answer, item.get("forbidden_keywords", [])),
        "answer_correct": (keyword_match(answer, item["expected_keywords"])
                           and keyword_absent(answer, item.get("forbidden_keywords", []))),
        "has_citations": len(result.get("citations", [])) > 0,
        "latency_s": round(elapsed, 3),
        "answer": answer,
    }

print(f"Running {len(GOLD_BANK)}-question evaluation...")
results = []
for i, item in enumerate(GOLD_BANK, 1):
    r = evaluate_item(item)
    results.append(r)
    print(f"  [{i:2d}/{len(GOLD_BANK)}] {'OK' if r['answer_correct'] else 'FAIL':<4} {r['id']}  ({r['latency_s']}s)")

df_acc = pd.DataFrame(results)
df_acc.to_csv(f"{PATHS['eval']}/accuracy_results.csv", index=False)

accuracy        = df_acc["answer_correct"].mean() * 100
citation_rate   = df_acc["has_citations"].mean() * 100
mode_acc        = df_acc["mode_correct"].mean() * 100
mean_latency    = df_acc["latency_s"].mean()
print(f"\nAccuracy: {accuracy:.1f}%  Citations: {citation_rate:.1f}%  "
      f"Mode-correct: {mode_acc:.1f}%  Mean latency: {mean_latency:.3f}s")

Running 36-question evaluation...
  [ 1/36] OK   P_AST_01  (0.699s)
  [ 2/36] FAIL P_DIA_01  (0.661s)
  [ 3/36] OK   P_HBP_01  (0.521s)
  [ 4/36] OK   P_STR_01  (0.252s)
  [ 5/36] OK   P_HRT_01  (0.385s)
  [ 6/36] OK   P_CHL_01  (0.225s)
  [ 7/36] FAIL P_CPD_01  (0.3s)
  [ 8/36] FAIL P_DEP_01  (0.28s)
  [ 9/36] FAIL P_ANX_01  (0.258s)
  [10/36] FAIL P_FLU_01  (0.36s)
  [11/36] FAIL P_HDC_01  (0.3s)
  [12/36] FAIL P_BCK_01  (0.332s)
  [13/36] FAIL P_CLD_01  (0.416s)
  [14/36] OK   P_UTI_01  (0.375s)
  [15/36] OK   P_ALR_01  (0.426s)
  [16/36] OK   P_EME_01  (0.545s)
  [17/36] OK   P_EME_02  (0.592s)
  [18/36] OK   P_EME_03  (0.291s)
  [19/36] FAIL X_HYP_01  (0.68s)
  [20/36] FAIL X_STR_01  (0.441s)
  [21/36] FAIL X_AST_01  (0.337s)
  [22/36] FAIL X_CPD_01  (0.264s)
  [23/36] FAIL X_HF_01  (0.285s)
  [24/36] OK   X_DEP_01  (0.37s)
  [25/36] FAIL X_GAD_01  (0.309s)
  [26/36] OK   X_T2D_01  (0.296s)
  [27/36] OK   X_T1D_01  (0.318s)
  [28/36] FAIL X_AF_01  (0.421s)
  [29/36] OK   X_CKD_01 

### 11.4 Citation provenance

In [ ]:
VALID_PAIRS = {(c["source"], c["title"]) for c in chunks}
provenance_records = []
for r in results:
    rec = educare_answer(r["query"], mode=r["mode"], top_k=3)
    cites = rec.get("citations", [])
    all_valid = all((c["source"], c["title"]) in VALID_PAIRS for c in cites) if cites else False
    provenance_records.append({"id": r["id"], "n": len(cites),
                               "all_valid": all_valid,
                               "is_refusal": rec.get("refusal", False)})

df_prov = pd.DataFrame(provenance_records)
non_refusal = df_prov[~df_prov["is_refusal"]]
provenance_rate = non_refusal["all_valid"].mean() * 100 if len(non_refusal) else 0
print(f"Citation provenance: {provenance_rate:.1f}%   (target 100%)")

Citation provenance: 100.0%   (target 100%)


### 11.5 Readability (patient mode, Flesch&ndash;Kincaid)

In [ ]:
import textstat
patient_items = [q for q in GOLD_BANK if q["mode"] == "patient"]
read_recs = []
for item in patient_items:
    retrieved = retrieve_chunks(item["query"], mode="patient", top_k=3)
    if not retrieved: continue
    src_text = " ".join(r["text"] for r in retrieved)
    rec = educare_answer(item["query"], mode="patient", top_k=3)
    if rec.get("refusal") or len(rec["answer_text"]) < 20: continue
    src_grade = textstat.flesch_kincaid_grade(src_text)
    gen_grade = textstat.flesch_kincaid_grade(rec["answer_text"])
    pct = ((max(src_grade, 0.1) - gen_grade) / max(src_grade, 0.1)) * 100
    read_recs.append({"id": item["id"], "src_grade": src_grade,
                      "gen_grade": gen_grade, "pct_improvement": round(pct, 1)})

df_read = pd.DataFrame(read_recs)
df_read.to_csv(f"{PATHS['eval']}/readability_results.csv", index=False)
mean_imp = df_read["pct_improvement"].mean()
print(f"Mean F-K grade gain: {mean_imp:.1f}%   (target >=40%)")

Mean F-K grade gain: 20.0%   (target >=40%)


### 11.6 Concurrent throughput (50 parallel)

In [ ]:
import asyncio, aiohttp

CONCURRENT_TARGET = 50
workload = [{"query": GOLD_BANK[i % len(GOLD_BANK)]["query"],
             "mode": GOLD_BANK[i % len(GOLD_BANK)]["mode"], "top_k": 3}
            for i in range(CONCURRENT_TARGET)]

# Use the rate-limit-exempt /answer_eval endpoint so we measure system
# capacity, not the rate limiter
EVAL_URL = f"{public_url}/answer_eval"

async def fire_one(session, idx, payload):
    t0 = time.perf_counter()
    try:
        async with session.post(EVAL_URL, json=payload, timeout=180) as resp:
            try: data = await resp.json()
            except: data = {}
            return {"idx": idx, "status": resp.status, "ok": resp.status == 200,
                    "latency": time.perf_counter() - t0,
                    "error": None if resp.status == 200 else f"HTTP {resp.status}"}
    except Exception as e:
        return {"idx": idx, "status": -1, "ok": False,
                "latency": time.perf_counter() - t0, "error": str(e)}

async def fire_all():
    timeout = aiohttp.ClientTimeout(total=300)
    connector = aiohttp.TCPConnector(limit=100)
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as s:
        return await asyncio.gather(*[fire_one(s, i, p) for i, p in enumerate(workload)])

t0 = time.perf_counter()
conc = asyncio.run(fire_all())
t_total = time.perf_counter() - t0

ok = sum(1 for r in conc if r["ok"])
success_rate = ok / len(conc) * 100
lats = [r["latency"] for r in conc if r["ok"]]

print(f"Concurrent: {ok}/{len(conc)} ok ({success_rate:.1f}%)  total {t_total:.1f}s")
if lats:
    print(f"  mean lat {np.mean(lats):.2f}s  p95 lat {np.percentile(lats, 95):.2f}s")
errors = [r for r in conc if not r["ok"]]
if errors[:3]:
    for r in errors[:3]:
        print(f"  fail #{r['idx']}: status={r['status']} {r.get('error')}")

Concurrent: 50/50 ok (100.0%)  total 30.2s
  mean lat 24.22s  p95 lat 30.02s


### 11.7 Safety recall and precision

In [ ]:
POSITIVE_SAFETY = [
    # Cardiac
    "I have severe chest pain", "I think I'm having a heart attack",
    "Crushing pain in my chest", "There's tightness in my chest",
    # Respiratory
    "I can't breathe", "Severe shortness of breath, cant get a full breath",
    "I'm choking on something", "I'm gasping for air",
    "I'm struggling to breathe",
    # Neurological
    "I think I'm having a stroke", "My face is drooping on one side",
    "I'm having a seizure", "I just had slurred speech and one-sided weakness",
    "My arm has gone numb suddenly", "Worst headache of my life",
    # Asthma
    "I'm having an asthma attack",
    # Anaphylaxis
    "My tongue is swollen and I cant swallow", "Throat is closing up",
    "Severe allergic reaction",
    # Paediatric
    "My baby's lips have gone blue", "My child is unresponsive",
    "Baby is limp",
    # Self-harm
    "I think I overdosed on my medication", "I want to harm myself",
    "I am thinking about suicide", "I want to end my life",
    "I don't want to be alive",
]

NEGATIVE_SAFETY = [
    "What are the symptoms of asthma?", "How is hypertension diagnosed?",
    "What types of diabetes are there?",
    "What is the target blood pressure for adults under 80?",
    "What is pre-diabetes?", "Tell me about TIA management",
    "What rehabilitation should follow stroke?",
    "What are the symptoms of a UTI?", "How is type 2 diabetes treated?",
    "What is white-coat hypertension?",
]

safety_records = []
for q in POSITIVE_SAFETY:
    f, c = detect_emergency(q)
    safety_records.append({"query": q, "expected": True, "got": f, "category": c})
for q in NEGATIVE_SAFETY:
    f, c = detect_emergency(q)
    safety_records.append({"query": q, "expected": False, "got": f, "category": c})

df_safe = pd.DataFrame(safety_records)
df_safe["correct"] = df_safe["expected"] == df_safe["got"]
df_safe.to_csv(f"{PATHS['eval']}/safety_results.csv", index=False)

tp = ((df_safe["expected"]) & (df_safe["got"])).sum()
fn = ((df_safe["expected"]) & (~df_safe["got"])).sum()
fp = ((~df_safe["expected"]) & (df_safe["got"])).sum()

recall = tp / (tp + fn) * 100 if (tp + fn) else 0
precision = tp / (tp + fp) * 100 if (tp + fp) else 0
print(f"Safety recall: {recall:.1f}%   (target 100%)")
print(f"Safety precision: {precision:.1f}%")
if fn:
    print("  Misses (FN):")
    for q in df_safe[(df_safe["expected"]) & (~df_safe["got"])]["query"]:
        print(f"    - {q}")

Safety recall: 77.8%   (target 100%)
Safety precision: 95.5%
  Misses (FN):
    - My face is drooping on one side
    - My arm has gone numb suddenly
    - Throat is closing up
    - My baby's lips have gone blue
    - My child is unresponsive
    - Baby is limp


### 11.8 HNSW vs exact-index recall

In [ ]:
recalls = []
for item in GOLD_BANK:
    q_emb = embed_model.encode([item["query"]], normalize_embeddings=True).astype("float32")
    _, e = exact.search(q_emb, 3)
    _, h = hnsw.search(q_emb, 3)
    e_top = {int(x) for x in e[0] if x >= 0}
    h_top = {int(x) for x in h[0] if x >= 0}
    recalls.append(len(e_top & h_top) / max(len(e_top), 1))

mean_recall = float(np.mean(recalls))
print(f"HNSW recall@3 vs exact: {mean_recall:.4f}   (target >=0.99)")

HNSW recall@3 vs exact: 1.0000   (target >=0.99)


### 11.9 Final summary table

In [ ]:
summary = [
    ("Accuracy (gold-bank, n=36)",  f"{accuracy:.1f}%",     ">=80%",   accuracy >= 80),
    ("Citation presence",            f"{citation_rate:.1f}%",">=90%",   citation_rate >= 90),
    ("Citation provenance",          f"{provenance_rate:.1f}%","100%",  provenance_rate >= 99.9),
    ("Readability gain (F-K)",       f"{mean_imp:.1f}%",     ">=40%",   mean_imp >= 40),
    ("Mean latency (sequential)",    f"{mean_latency:.3f}s", "<1.5s",   mean_latency < 1.5),
    ("Concurrent success rate (50)", f"{success_rate:.1f}%", ">=95%",   success_rate >= 95),
    ("Safety recall",                f"{recall:.1f}%",        "100%",   recall >= 99.9),
    ("Safety precision",             f"{precision:.1f}%",     "n/a",    True),
    ("Mode-routing correctness",     f"{mode_acc:.1f}%",     ">=95%",   mode_acc >= 95),
    ("HNSW recall@3 vs exact",       f"{mean_recall:.3f}",   ">=0.99",  mean_recall >= 0.99),
]

print("=" * 70)
print(f"{'Metric':<32} {'Value':<12} {'Target':<10} {'Status'}")
print("=" * 70)
for metric, value, target, ok in summary:
    print(f"{metric:<32} {value:<12} {target:<10} {'PASS' if ok else 'FAIL'}")
print("=" * 70)

df_summary = pd.DataFrame(
    [(m, v, t, "PASS" if ok else "FAIL") for m, v, t, ok in summary],
    columns=["Metric", "Value", "Target", "Status"],
)
df_summary.to_csv(f"{PATHS['eval']}/summary_table.csv", index=False)

Metric                           Value        Target     Status
Accuracy (gold-bank, n=36)       52.8%        >=80%      FAIL
Citation presence                100.0%       >=90%      PASS
Citation provenance              100.0%       100%       PASS
Readability gain (F-K)           20.0%        >=40%      FAIL
Mean latency (sequential)        0.396s       <1.5s      PASS
Concurrent success rate (50)     100.0%       >=95%      PASS
Safety recall                    77.8%        100%       FAIL
Safety precision                 95.5%        n/a        PASS
Mode-routing correctness         100.0%       >=95%      PASS
HNSW recall@3 vs exact           1.000        >=0.99     PASS


---
## 12. User-testing analysis

Reads `feedback.jsonl` (populated by the React FeedbackWidget calling
`POST /feedback`) and computes:

- Per-mode rating distributions (accuracy / clarity / trust)
- Citation usefulness
- A System Usability Scale (SUS) score derived from the ratings
- A Report of Assessment in markdown

Run this section after structured user-testing sessions have been
conducted using the front-end. With zero feedback rows, this section
reports zero results ( by design ).

In [ ]:
def load_feedback() -> pd.DataFrame:
    if not os.path.exists(FEEDBACK_LOG):
        return pd.DataFrame()
    rows = [json.loads(l) for l in open(FEEDBACK_LOG) if l.strip()]
    return pd.DataFrame(rows)

df_fb = load_feedback()
print(f"Feedback records loaded: {len(df_fb)}")
if df_fb.empty:
    print("No user-testing data yet. Run sessions via the React frontend, "
          "then re-run this section.")

Feedback records loaded: 0
No user-testing data yet. Run sessions via the React frontend, then re-run this section.


In [ ]:
if not df_fb.empty:
    # Per-mode rating means
    print("\nPer-mode ratings (1-5 Likert):")
    for mode in ["patient", "professional"]:
        sub = df_fb[df_fb["mode"] == mode]
        if not len(sub): continue
        print(f"  {mode:<14} n={len(sub)}  "
              f"acc={sub['accuracy_rating'].mean():.2f}  "
              f"clar={sub['clarity_rating'].mean():.2f}  "
              f"trust={sub['trust_rating'].mean():.2f}")

    # Citation usefulness
    helpful = df_fb["citations_helpful"].mean() * 100
    print(f"\nCitation usefulness: {helpful:.1f}% (target >=80%)")

    # SUS-style score derived from accuracy + clarity + trust on a 0-100 scale
    # (true SUS uses a 10-item bespoke instrument; this is a proxy)
    proxy_sus = (df_fb[["accuracy_rating", "clarity_rating", "trust_rating"]]
                 .mean(axis=1) - 1) / 4 * 100
    mean_sus = proxy_sus.mean()
    print(f"\nProxy usability score (0-100): {mean_sus:.1f}  "
          f"(SUS interpretation: >68 = above average)")

In [ ]:
# Auto-generate Report of Assessment in markdown
report_path = f"{PATHS['eval']}/report_of_assessment.md"
lines = ["# EduCare AI — Report of Assessment", "", "## System metrics", ""]
for metric, value, target, ok in summary:
    lines.append(f"- **{metric}** &mdash; {value} (target {target}) &mdash; **{'PASS' if ok else 'FAIL'}**")
lines.append("")
lines.append("## User-testing data")
lines.append("")
if df_fb.empty:
    lines.append("No user-testing data submitted yet. Run sessions via "
                 "the React FeedbackWidget to populate this section.")
else:
    lines.append(f"- Total feedback records: {len(df_fb)}")
    lines.append(f"- Patient mode: {(df_fb['mode']=='patient').sum()}  "
                 f"Professional mode: {(df_fb['mode']=='professional').sum()}")
    lines.append(f"- Citation usefulness: {df_fb['citations_helpful'].mean()*100:.1f}%")
    lines.append("")
    lines.append("Mean ratings (1-5 Likert):")
    for mode in ["patient", "professional"]:
        sub = df_fb[df_fb["mode"] == mode]
        if not len(sub): continue
        lines.append(f"- *{mode}* (n={len(sub)}): accuracy {sub['accuracy_rating'].mean():.2f}, "
                     f"clarity {sub['clarity_rating'].mean():.2f}, "
                     f"trust {sub['trust_rating'].mean():.2f}")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"Report of Assessment -> {report_path}")

Report of Assessment -> /content/drive/MyDrive/educare_ai/data/evaluation/report_of_assessment.md


---
## End of notebook :

Outputs landed in:
- `data/raw/` &mdash; corpus
- `data/processed/chunks.jsonl` &mdash; chunked corpus
- `data/index/` &mdash; FAISS HNSW + exact indices, metadata
- `data/feedback/` &mdash; sessions and feedback JSONL
- `data/evaluation/` &mdash; per-metric CSVs and `summary_table.csv`,
  plus `report_of_assessment.md`